# I HATE PHENIKAA — V18 SMART RANDOM PROXY — STRICT + FALLBACK

Phase 10 NLU + YOLO + V18 frozen. Strict unchanged; fallback randomly substitutes missing named landmark types from detected map with fixed seed.


## 1. Install dependencies & mount Drive

In [1]:
%pip install -q ultralytics "transformers<5" sentencepiece protobuf "numpy<3" numba scipy lightgbm tqdm ijson
from google.colab import drive
drive.mount("/content/drive")
!unzip -q -o "/content/drive/MyDrive/Phenikaa_Campus_Courier_2026_v3.zip" -d "/content/dataset"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingf

## 2. Paths, models and CPU/GPU auto-selection

In [2]:
import os, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'; os.environ['YOLO_VERBOSE'] = 'False'; os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
import re, json, glob, math, heapq, shutil, unicodedata, collections, traceback
from pathlib import Path
import cv2
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from functools import partial
from tqdm.auto import tqdm
tqdm = partial(tqdm, disable=True)      # disable inference progress output
from transformers import AutoTokenizer, AutoModel
import transformers; transformers.logging.set_verbosity_error()

# ---------------- ĐƯỜNG DẪN ----------------
ROOT      = "/content/dataset/Phenikaa_Campus_Courier_2026_v3/delivery_public"
TEST_DIR  = f"{ROOT}/test"
VAL_DIR   = f"{ROOT}/validation"

YOLO_PATH = "/content/drive/MyDrive/YOLO_Courier_Backup_M/best.pt"
NLU_P10_CKPT = "/content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/bamibert_nlu_phase12_v3_4views_best.pt"
NLU_V7_CKPT = "/content/drive/MyDrive/bamibert_nlu_v7_from_ft4_best.pt"
NLU_V6_CKPT = "/content/drive/MyDrive/bamibert_nlu_v6_best.pt"
# Strict Phase10: do not silently use an older checkpoint.
NLU_CKPT = NLU_P10_CKPT

# Upload model bundle ZIP (provided separately) to this location.
ROBOT_V18_ZIP = "/content/drive/MyDrive/fixv18/fixfallback18/COURIER_V3_V18_EMPTY_GUARDED_RUNTIME.zip"
ROBOT_V18_DIR = "/content/robot_v18_empty_guarded_runtime"
ROBOT_V18_BATCH = 256
RESIZED_DIR     = "/content/pkaa_v18_resized"       # ảnh test sau khi resize 1024
OUT_SUBMISSION  = "/content/submission_fallback.json"
OUT_FLAT        = "/content/submission_fallback_flat.json"      # bản dự phòng: list phẳng 10 số / scene (giống labels.json)
DRIVE_OUT_DIR   = "/content/drive/MyDrive"             # copy kết quả lên Drive (lỗi thì bỏ qua)

# ---------------- THAM SỐ ĐÃ CHỐT (đúng như notebook gốc) ----------------
IMGSZ         = 1024      # kích thước lúc train YOLO
YOLO_MIN_CONF = 0.1       # collect_dets(min_conf=0.1) rồi lọc lại bằng conf=0.4 (đường chạy đạt 300/300 val)
YOLO_CONF     = 0.5
MODEL_NAME    = "Qualcomm-AI-Research/BamiBERT"
MAX_LEN       = 256
USE_TTA       = False     # đo trên val: TTA không giúp -> tắt (cell suy luận NLU gốc in ra USE_TTA = False)

# ---------------- KHOÁ TRONG observations.json (None = tự dò) ----------------
OBS_ID_KEY = None         # vd 'scene_id' / 'id'
OBS_IMAGE_KEY = None      # vd 'image'
OBS_TEXT_KEY = None       # vd 'mission_text' / 'text'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

# Khởi tạo theo batch, cache tiến độ, đọc observations một lần
YOLO_BATCH = 8          # giảm nếu GPU hết VRAM
NLU_BATCH = 48          # giảm nếu GPU hết VRAM
ROBOT_V18_BATCH = 256
CACHE_DIR = Path('/content/pkaa_v18_cache')
DEBUG_LIMIT = None       # 5 để smoke-test; None để chạy toàn bộ test
STRICT_SUBMISSION = False   # both results saved; -1 allowed only in strict/debug files
CACHE_DIR.mkdir(parents=True,exist_ok=True)

# GPU OPTIONAL. CPU may be slow; no hard stop, no CUDA preflight.
YOLO_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
YOLO_BATCH = 8 if torch.cuda.is_available() else 2
print('[DEVICE] NLU:',device,'YOLO:',YOLO_DEVICE,'batch:',YOLO_BATCH)


[DEVICE] NLU: cuda YOLO: 0 batch: 8


## 3. Frozen Courier V18 runtime (including R2 strict tie-break and R9 greedy)

In [3]:
# Exact frozen V18 inference runtime; no training or labels in this distribution.
import zipfile,sys,contextlib,io
from pathlib import Path
_runtime_root=Path(ROBOT_V18_DIR)
_runtime_root.mkdir(parents=True,exist_ok=True)
_runtime_manifest=_runtime_root/'production'/'frozen_interpretable_weights.json'
if not _runtime_manifest.is_file():
    if not Path(ROBOT_V18_ZIP).is_file():
        raise FileNotFoundError(f'Missing V18 runtime: {ROBOT_V18_ZIP}. Upload the bundled runtime zip to Drive.')
    with zipfile.ZipFile(ROBOT_V18_ZIP) as z:
        for member in z.infolist():
            path=Path(member.filename)
            if path.is_absolute() or '..' in path.parts: raise ValueError('Unsafe ZIP member: '+member.filename)
            z.extract(member,_runtime_root)
for folder in (_runtime_root, _runtime_root/'production'):
    if str(folder) not in sys.path:sys.path.insert(0,str(folder))
if not _runtime_manifest.is_file() or not (_runtime_root/'r2_lex_research.py').is_file():
    raise FileNotFoundError('Incomplete V18 runtime: manifest or R2 lex module missing')
from courier10_transfer_v18 import predict_all as robot_v18_predict_all
def predict_v18_batch(scenes):
    # suppress internal per-robot printed lines only; exceptions remain visible to caller
    with contextlib.redirect_stdout(io.StringIO()):
        return robot_v18_predict_all(scenes,str(_runtime_root/'production'))
print('V18 ready:',_runtime_root,'| CPU/GPU optional')


V18 ready: /content/robot_v18_empty_guarded_runtime | CPU/GPU optional


## 4. Phase 10 NLU — exact original 8-head schema & constrained decoder

In [4]:
# NEW NLU HEAD SCHEMA — unchanged 8 heads; Phase10 source schema
PLACES = ['canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports']
DIRECT_KINDS = ('none','near','far','north','south','east','west')
SPATIAL_KINDS = ('anchor_near','north_most','south_most','east_most','west_most')
KINDS = list(DIRECT_KINDS + SPATIAL_KINDS)
HEADS = {
    'goal': PLACES + ['none'],                  # APPEND at index 10 (old ten indices unchanged)
    'via': ['none'] + PLACES,                  # 'none' shared: absent OR map-based VIA, disambiguated by VKIND
    'gkind': KINDS,
    'ganchor': ['none'] + PLACES,
    'vkind': KINDS,
    'vanchor': ['none'] + PLACES,
    'urgent': ['no','yes'],
    'fragile': ['no','yes'],
}
H2I = {h:{l:i for i,l in enumerate(ls)} for h,ls in HEADS.items()}
I2H = {h:{i:l for i,l in enumerate(ls)} for h,ls in HEADS.items()}
HEAD_W = {'goal':2.,'via':2.,'gkind':1.5,'ganchor':1.3,'vkind':1.1,'vanchor':1.1,'urgent':1.,'fragile':1.}

def strip_accents(t):
    t = t.replace('đ','d').replace('Đ','D')
    return ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
def fold(s): return strip_accents(s).lower()

# Joint constrained decoding for BOTH roles.
# GOAL: [named place & direct gkind] OR [none & spatial gkind].
# VIA:  [none & no vkind] OR [named place & direct vkind] OR [none & spatial vkind].
# This keeps the checkpoint head shapes unchanged. NLU V7 needs spatial-VIA examples.

def decode(P, rules=None):
    rules = RULES if rules is None else rules
    L = {h: np.log(np.clip(np.asarray(P[h], dtype=np.float64), 1e-12, 1.0)) for h in HEADS}
    N = len(L['goal'])
    out = {h:np.zeros(N,dtype=np.int64) for h in HEADS}
    for h in ('urgent','fragile'):
        out[h] = L[h].argmax(axis=1)

    def best_reference(kind_log, anchor_log, own_place, allowed):
        best = (-np.inf, H2I['gkind']['none'], 0)
        for kind in allowed:
            ki=H2I['gkind'][kind]
            if kind in ('near','far','anchor_near'):
                valid = [j for j in range(1,len(PLACES)+1)
                         if kind=='anchor_near' or not rules['anchor_ne_own'] or j!=own_place]
                if not valid: continue
                anc=max(valid,key=lambda j:anchor_log[j])
            else:
                anc=0
            score=float(kind_log[ki]+anchor_log[anc])
            if score>best[0]:best=(score,ki,anc)
        return best

    for i in range(N):
        gopts = {}
        for gi,goal in enumerate(HEADS['goal']):
            allowed=SPATIAL_KINDS if goal=='none' else DIRECT_KINDS
            sk,ki,ai=best_reference(L['gkind'][i],L['ganchor'][i],
                                    gi+1 if goal!='none' else 0,allowed)
            gopts[gi]=(float(L['goal'][i,gi]+sk),ki,ai)
        vopts={}
        for vi,via in enumerate(HEADS['via']):
            if via=='none':
                # Candidate A: no waypoint at all.
                ka=H2I['vkind']['none']
                score_a=float(L['via'][i,vi]+L['vkind'][i,ka]+L['vanchor'][i,0])
                # Candidate B: waypoint exists, type determined by map, via head still 'none'.
                score_b,kb,ab=best_reference(L['vkind'][i],L['vanchor'][i],0,SPATIAL_KINDS)
                score_b=float(L['via'][i,vi]+score_b)
                vopts[vi]=max((score_a,ka,0),(score_b,kb,ab), key=lambda x:x[0])
            else:
                sc,ki,ai=best_reference(L['vkind'][i],L['vanchor'][i],vi,DIRECT_KINDS)
                vopts[vi]=(float(L['via'][i,vi]+sc),ki,ai)
        best=None
        for gi,goal in enumerate(HEADS['goal']):
            for vi,via in enumerate(HEADS['via']):
                if rules['goal_ne_via'] and goal!='none' and via!='none' and goal==via:
                    continue
                score=gopts[gi][0]+vopts[vi][0]
                if best is None or score>best[0]:best=(score,gi,vi)
        assert best is not None
        _,gi,vi=best
        out['goal'][i],out['via'][i]=gi,vi
        out['gkind'][i],out['ganchor'][i]=gopts[gi][1:]
        out['vkind'][i],out['vanchor'][i]=vopts[vi][1:]
        # No phantom waypoint, and map-qualified via is represented by vkind even when via=none.
        assert (gi==H2I['goal']['none']) == (HEADS['gkind'][out['gkind'][i]] in SPATIAL_KINDS)
        if vi==H2I['via']['none']:
            assert HEADS['vkind'][out['vkind'][i]] in ('none',)+SPATIAL_KINDS
    return out

RULES = {'goal_ne_via':True,'anchor_ne_own':True}


In [5]:
class MultiHeadNLU(nn.Module):
    def __init__(self, name, extra_pools=3):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        hidden = self.enc.config.hidden_size
        self.extra_pools = extra_pools
        self.drop = nn.Dropout(0.2)
        self.pool_q = nn.ModuleDict({
            head: nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
            for head in HEADS
        })
        self.heads = nn.ModuleDict({head: nn.Linear(hidden * 2, len(labels)) for head, labels in HEADS.items()})
        self.extra_pool_q = nn.ModuleDict({head: nn.Linear(hidden, extra_pools, bias=False) for head in HEADS})
        extra_size = hidden * (extra_pools + 1)
        self.extra_norm = nn.ModuleDict({head: nn.LayerNorm(extra_size) for head in HEADS})
        self.extra_heads = nn.ModuleDict({head: nn.Linear(extra_size, len(labels)) for head, labels in HEADS.items()})
        for head in HEADS:
            nn.init.zeros_(self.pool_q[head][2].weight)
            nn.init.zeros_(self.pool_q[head][2].bias)
            nn.init.normal_(self.extra_pool_q[head].weight, mean=0.0, std=0.02)
            nn.init.zeros_(self.extra_heads[head].weight)
            nn.init.zeros_(self.extra_heads[head].bias)

    def forward(self, ids, mask, return_emb=False):
        hidden = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        cls = hidden[:, 0, :]; logits = {}; token_mask = mask.unsqueeze(-1) == 0
        for head in HEADS:
            weight = self.pool_q[head](hidden).squeeze(-1).float().masked_fill(mask == 0, -1e4).softmax(-1).to(hidden.dtype)
            pooled = (weight.unsqueeze(-1) * hidden).sum(1)
            old_logits = self.heads[head](self.drop(torch.cat([cls, pooled], -1)))

            extra_score = self.extra_pool_q[head](hidden).float().masked_fill(token_mask, -1e4)
            extra_weight = extra_score.softmax(dim=1).to(hidden.dtype)
            extra_pool = torch.einsum('btk,bth->bkh', extra_weight, hidden).flatten(1)
            extra_feature = self.extra_norm[head](torch.cat([cls, extra_pool], -1))
            logits[head] = old_logits + self.extra_heads[head](self.drop(extra_feature))

        if not return_emb: return logits
        mask3 = mask.unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * mask3).sum(1) / mask3.sum(1).clamp(min=1)
        emb = F.normalize(F.normalize(cls.float(), dim=-1) + F.normalize(mean.float(), dim=-1), dim=-1)
        return logits, emb

@torch.no_grad()
def predict_probs(model, texts, bs=64):
    model.eval(); texts = list(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i])); chunks = {h: [] for h in HEADS}
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            lg = model(enc['input_ids'], enc['attention_mask'])
        for h in HEADS: chunks[h].append(lg[h].float().softmax(-1).cpu().numpy())
    inv = np.empty(len(order), dtype=int); inv[np.array(order)] = np.arange(len(order))
    return {h: np.concatenate(chunks[h])[inv] for h in HEADS}

_BRACKET = re.compile(r'\[[^\]]*\]')
def tta_views(t):
    base = re.sub(r'\s+', ' ', _BRACKET.sub(' ', t)).strip() or t      # bỏ cụm [Đơn #1234]
    return [t, base, strip_accents(base)]
TTA_W = (2.0, 1.0, 1.0)
def predict_probs_tta(model, texts):
    acc = None
    for vi, w in enumerate(TTA_W):
        P = predict_probs(model, [tta_views(t)[vi] for t in texts])
        acc = {h: w * P[h] for h in HEADS} if acc is None else {h: acc[h] + w * P[h] for h in HEADS}
    return {h: v / sum(TTA_W) for h, v in acc.items()}


def parse_missions(texts, tta=None):
    tta = USE_TTA if tta is None else tta
    P = predict_probs_tta(model,texts) if tta else predict_probs(model,texts)
    pred = decode(P)
    out=[]
    for i in range(len(texts)):
        p={h:HEADS[h][int(pred[h][i])] for h in HEADS}
        def ref(k,a):
            return None if p[k]=='none' else {
                'kind':p[k],
                'anchor':None if p[a]=='none' else p[a]
            }
        m={
            'goal':p['goal'],
            'goal_ref':ref('gkind','ganchor'),
            'via':None if p['via']=='none' else p['via'],
            'via_ref':ref('vkind','vanchor'),  # VIA map kind preserved even when VIA head is 'none'
            'urgent':p['urgent']=='yes',
            'fragile':p['fragile']=='yes',
            'via_required': (p['via']!='none' or p['vkind'] in SPATIAL_KINDS),
            'min_conf':min(float(P[h][i].max()) for h in HEADS)
        }
        # The location head may be noisy; constrained decoding guarantees this invariant.
        assert (m['goal']=='none') == (p['gkind'] in SPATIAL_KINDS)
        out.append(m)
    return out

# Phase10: exact label order is critical, including GOAL/ VIA 'none' positions.
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
assert os.path.isfile(NLU_CKPT), f'Missing Phase10 NLU checkpoint: {NLU_CKPT}'
_ck = torch.load(NLU_CKPT, map_location='cpu', weights_only=False)
assert _ck.get('phase') in (10,11,12), f'Wrong checkpoint phase: {_ck.get("phase")}'
assert _ck.get('heads') == HEADS, 'NLU 8-head label order differs from saved Phase10 model'
for head, labels in HEADS.items():
    assert _ck['state'][f'heads.{head}.weight'].shape[0] == len(labels), head
model = MultiHeadNLU(MODEL_NAME).to(device)
model.load_state_dict(_ck['state'], strict=True)
model.eval()
print('Loaded Phase10 NLU:', NLU_CKPT, '| best_epoch:', _ck.get('epoch', 'unknown'))
print('Heads:', {h:len(x) for h,x in HEADS.items()})
del _ck


Loaded Phase10 NLU: /content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/bamibert_nlu_phase12_v3_4views_best.pt | best_epoch: 2
Heads: {'goal': 11, 'via': 11, 'gkind': 12, 'ganchor': 11, 'vkind': 12, 'vanchor': 11, 'urgent': 2, 'fragile': 2}


### NLU smoke examples (diagnostic only, not accuracy metric)

In [6]:
def phase10_semantic_smoke():
    checks=[
      ('Giao tới thư viện gần KTX.', {'goal':'library','gkind':'near','ganchor':'dorm'}),
      ('Giao tới địa điểm gần KTX nhất.', {'goal':'none','gkind':'anchor_near','ganchor':'dorm'}),
      ('Ghé phòng khám gần cổng trước, rồi giao tại thư viện.',
       {'via':'clinic','vkind':'near','vanchor':'gate','goal':'library'}),
      ('Ghé địa điểm gần cổng nhất trước, rồi giao tại thư viện.',
       {'via':'none','vkind':'anchor_near','vanchor':'gate','goal':'library'}),
      ('Ghé phòng khám trước, rồi giao ở thư viện gần KTX.',
       {'via':'clinic','vkind':'none','goal':'library','gkind':'near','ganchor':'dorm'}),
      ('Giao ở thư viện gần KTX; nhưng phải ghé phòng khám gần cổng trước.',
       {'goal':'library','gkind':'near','ganchor':'dorm','via':'clinic','vkind':'near','vanchor':'gate'}),
      ('Giao tới địa điểm cực Đông.', {'goal':'none','gkind':'east_most'}),
      ('Ghé địa điểm cực Bắc trước, rồi giao ở phòng khám.',
       {'goal':'clinic','via':'none','vkind':'north_most'}),
    ]
    pred=parse_missions([t for t,_ in checks],tta=False)
    ok=0
    for (txt,y),p in zip(checks,pred):
        flattened={}
        for role,abbr in (('goal','g'),('via','v')):
            ref=p.get(role+'_ref') or {}
            flattened[role]=p.get(role) or 'none'
            flattened[abbr+'kind']=ref.get('kind','none')
            flattened[abbr+'anchor']=ref.get('anchor') or 'none'
        wrong={k:(v,flattened.get(k)) for k,v in y.items() if flattened.get(k)!=v}
        ok+=int(not wrong)
        print(('OK ' if not wrong else 'CHECK '),txt,' | ',wrong or 'heads match')
    print(f'Phase10 semantic smoke: {ok}/{len(checks)}. Synthetic smoke is diagnostic, not validation accuracy.')
phase10_semantic_smoke()


OK  Giao tới thư viện gần KTX.  |  heads match
OK  Giao tới địa điểm gần KTX nhất.  |  heads match
OK  Ghé phòng khám gần cổng trước, rồi giao tại thư viện.  |  heads match
CHECK  Ghé địa điểm gần cổng nhất trước, rồi giao tại thư viện.  |  {'via': ('none', 'office'), 'vkind': ('anchor_near', 'near')}
OK  Ghé phòng khám trước, rồi giao ở thư viện gần KTX.  |  heads match
OK  Giao ở thư viện gần KTX; nhưng phải ghé phòng khám gần cổng trước.  |  heads match
OK  Giao tới địa điểm cực Đông.  |  heads match
CHECK  Ghé địa điểm cực Bắc trước, rồi giao ở phòng khám.  |  {'via': ('none', 'canteen'), 'vkind': ('north_most', 'north')}
Phase10 semantic smoke: 6/8. Synthetic smoke is diagnostic, not validation accuracy.


## 5. YOLO 38 classes, dynamic grid, edges, legend and graph

In [7]:
CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3',
    'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered'
]

EDGE_CLS = ('edge_type_', 'print_edge_')                  # box cạnh (kiểu nét) -> status lấy từ legend
EDGE_BOX_CLS = {'closed'}                                 # closed = box phủ cả đoạn đường
MODIFIER_CLS = {'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right'}  # icon nhỏ giữa cạnh
NODE_CLS = set(CLASSES[:14])                              # landmark + robot


# ==========================================
# HELPERS
# ==========================================
def get_center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)


def get_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def dynamic_cluster(coords, unit_size):
    if not coords:
        return []
    coords = sorted(coords)
    clusters = [[coords[0]]]
    tolerance = unit_size * 0.4
    for c in coords[1:]:
        if c - np.mean(clusters[-1]) <= tolerance:
            clusters[-1].append(c)
        else:
            clusters.append([c])
    return [float(np.mean(cl)) for cl in clusters]


def get_grid_index(val, clusters):
    return min(range(len(clusters)), key=lambda i: abs(clusters[i] - val))


def is_plain_edge(c):
    return c.startswith(EDGE_CLS)


# ==========================================
# LEGEND: swatch k -> txt bên cạnh -> status
# ==========================================
def build_legend(dets, txt_side='right', pairs_out=None):
    texts = [d for d in dets if d['class'].startswith('txt_')]
    swatches = [d for d in dets if d['class'].startswith(('swatch_edge_', 'swatch_print_'))]
    if not texts or not swatches:
        return {}

    def find_label(s):
        sx1, _, sx2, _ = s['box']
        slack = 0.3 * s['w']
        row = [t for t in texts
               if abs(t['center'][1] - s['center'][1]) <= max(s['h'], t['h']) * 0.6]
        if txt_side == 'left':
            side = [t for t in row if t['box'][2] <= sx1 + slack]
        else:
            side = [t for t in row if t['box'][0] >= sx2 - slack]
        pool = side or row or texts
        return min(pool, key=lambda t: get_distance(t['center'], s['center']))

    best = {}  # k -> (conf swatch, status)
    for s in swatches:
        k = s['class'][-1]
        t = find_label(s)
        if pairs_out is not None:
            pairs_out.append((s, t))
        status = t['class'].replace('txt_', '')
        if k not in best or s['conf'] > best[k][0]:
            best[k] = (s['conf'], status)

    legend = {}
    for k, (_, status) in best.items():
        legend['edge_type_' + k] = status
        legend['print_edge_' + k] = status
    return legend


def _oneway_target(direction, na, nb):
    if direction == 'up':
        return [min(na[0], nb[0]), na[1]]
    if direction == 'down':
        return [max(na[0], nb[0]), na[1]]
    if direction == 'left':
        return [na[0], min(na[1], nb[1])]
    return [na[0], max(na[1], nb[1])]  # right


# ==========================================
# 1. YOLO -> DETS (chạy model 1 lần, lọc ngưỡng sau)
# ==========================================
def predict_dets(model, image_path, min_conf=0.1):
    res = model.predict(image_path, conf=min_conf, imgsz=IMGSZ, verbose=False)[0]
    img = res.orig_img[:, :, ::-1]  # BGR -> RGB
    dets = []
    for box, cls_id, cf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
        box = [float(v) for v in box]
        dets.append({'class': CLASSES[int(cls_id)], 'box': box, 'conf': float(cf),
                     'center': get_center(box), 'w': box[2] - box[0], 'h': box[3] - box[1]})
    return dets, img


# ==========================================
# 2. DETS -> JSON GRAPH
#    box cạnh   : edge_type_k / print_edge_k (status = legend) / closed
#    icon nhỏ   : stairs / oneway_*  -> gán vào cạnh có box chứa tâm icon
#    split_long : box cạnh dài >1 ô -> tách thành nhiều cạnh 1 ô (False = giữ kiểu cũ: 1 cạnh nối xa)
# ==========================================
def dets_to_scene(dets, mission_via=None, mission_goal=None, conf=0.4, edge_conf=None,
                  txt_side='right', split_long=True, trace=None):
    edge_conf = conf if edge_conf is None else edge_conf
    dets = [dict(d) for d in dets
            if d['conf'] >= (edge_conf if is_plain_edge(d['class']) else conf)]

    pairs = []
    legend = build_legend(dets, txt_side=txt_side, pairs_out=pairs)
    for d in dets:
        if is_plain_edge(d['class']):
            d['status'] = legend.get(d['class'], '?')

    edge_dets = [d for d in dets if is_plain_edge(d['class']) or d['class'] in EDGE_BOX_CLS]
    modifiers = [d for d in dets if d['class'] in MODIFIER_CLS]

    # thước đo ô lưới: chỉ đo trên box cạnh (không tính icon)
    hw = [d['w'] for d in edge_dets if d['w'] > d['h']]
    vh = [d['h'] for d in edge_dets if d['h'] > d['w']]
    grid_w = float(np.median(hw)) if hw else 100.0
    grid_h = float(np.median(vh)) if vh else 100.0

    scene = {'nodes': [], 'edges': [], 'landmarks': [], 'robot': {},
             'weather': 'rain' if any(d['class'] == 'weather_rain' for d in dets) else 'dry',
             'mission': {'via': mission_via, 'goal': mission_goal}, 'warnings': []}

    # --- lưới động
    raw_x, raw_y = [], []
    for d in dets:
        if d['class'] in NODE_CLS:
            raw_x.append(d['center'][0])
            raw_y.append(d['center'][1])
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        if d['w'] >= d['h']:
            raw_y.append(d['center'][1])
            raw_x += [x1, x2]
        else:
            raw_x.append(d['center'][0])
            raw_y += [y1, y2]
    col_cl, row_cl = dynamic_cluster(raw_x, grid_w), dynamic_cluster(raw_y, grid_h)

    # --- landmark / robot
    for d in dets:
        if d['class'] not in NODE_CLS:
            continue
        r, c = get_grid_index(d['center'][1], row_cl), get_grid_index(d['center'][0], col_cl)
        if d['class'].startswith('robot_'):
            scene['robot'] = {'rc': [r, c], 'heading': d['class'].split('_')[1].upper()}
        else:
            scene['landmarks'].append({'type': d['class'], 'rc': [r, c]})
    for r in range(len(row_cl)):
        for c in range(len(col_cl)):
            scene['nodes'].append({'rc': [r, c]})

    # --- cạnh
    edges = {}
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        horiz = d['w'] >= d['h']
        if horiz:
            r = get_grid_index(d['center'][1], row_cl)
            ca, cb = sorted([get_grid_index(x1, col_cl), get_grid_index(x2, col_cl)])
            steps = [([r, c], [r, c + 1]) for c in range(ca, cb)]
        else:
            c = get_grid_index(d['center'][0], col_cl)
            ra, rb = sorted([get_grid_index(y1, row_cl), get_grid_index(y2, row_cl)])
            steps = [([r, c], [r + 1, c]) for r in range(ra, rb)]
        if not steps:
            continue
        if len(steps) > 1:
            scene['warnings'].append(f"{d['class']} dai {len(steps)} o: {steps[0][0]}->{steps[-1][1]}"
                                     f" ({'tach' if split_long else 'giu nguyen'})")
            if not split_long:
                steps = [(steps[0][0], steps[-1][1])]

        for na, nb in steps:
            if horiz:
                box = [col_cl[na[1]], y1, col_cl[nb[1]], y2]
            else:
                box = [x1, row_cl[na[0]], x2, row_cl[nb[0]]]
            key = (tuple(na), tuple(nb))
            e = edges.setdefault(key, {'a': na, 'b': nb, 'status': None, 'stairs': False,
                                       'oneway_to': None, '_box': box, '_horiz': horiz,
                                       '_center': get_center(box), '_plain_conf': -1.0,
                                       '_closed': False, '_src': None})
            cls = d['class']
            if is_plain_edge(cls):
                if d['conf'] > e['_plain_conf']:      # 2 box plain chồng nhau -> lấy box conf cao hơn
                    e['_plain_conf'] = d['conf']
                    e['status'] = legend.get(cls, 'normal')
                    e['_src'] = ('legend:' if cls in legend else 'NO_LEGEND:') + cls
            else:  # closed
                e['_closed'] = True

    for e in edges.values():
        if e['_closed']:
            e['status'] = 'closed'
            e['_src'] = 'closed_box'

    # --- icon nhỏ -> cạnh có box chứa tâm icon, đúng hướng
    tol = 0.15 * min(grid_w, grid_h)
    # --- icon nhỏ -> cạnh có diện tích đè (overlap) lớn nhất với icon
    for m in modifiers:
        mx1, my1, mx2, my2 = m['box']
        m_area = (mx2 - mx1) * (my2 - my1)
        direction = m['class'].split('_')[1] if m['class'].startswith('oneway_') else None

        best, max_overlap_area = None, 0.0

        for e in edges.values():
            ex1, ey1, ex2, ey2 = e['_box']

            # 1. Kiểm tra hướng tương thích trước khi xét diện tích đè
            if direction in ('left', 'right') and not e['_horiz']:
                continue
            if direction in ('up', 'down') and e['_horiz']:
                continue

            # 2. Tính diện tích phần giao nhau (Intersection) giữa box icon và box cạnh
            inter_x1 = max(mx1, ex1)
            inter_y1 = max(my1, ey1)
            inter_x2 = min(mx2, ex2)
            inter_y2 = min(my2, ey2)

            if inter_x2 > inter_x1 and inter_y2 > inter_y1:
                overlap_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)

                # Chọn cạnh có diện tích đè lớn nhất
                if overlap_area > max_overlap_area:
                    max_overlap_area = overlap_area
                    best = e

        # Nếu không có cạnh nào đè lên icon (diện tích giao = 0)
        if best is None or max_overlap_area == 0:
            scene['warnings'].append(f"icon {m['class']} tai {np.round(m['center']).tolist()} khong đè lên cạnh nào")
            continue

        if m['class'] == 'stairs':
            best['stairs'] = True
        else:
            best['oneway_to'] = _oneway_target(direction, best['a'], best['b'])

    if trace is not None:
        trace.update(dets=dets, legend=legend, pairs=pairs,
                     edge_src={k: e['_src'] for k, e in edges.items()})

    for e in edges.values():
        for k in ('_box', '_horiz', '_center', '_plain_conf', '_closed', '_src'):
            e.pop(k, None)
        scene['edges'].append(e)
    return scene


In [8]:
from ultralytics import YOLO
yolo = YOLO(YOLO_PATH)
class_names = yolo.names
actual = [str(class_names[i]) for i in range(len(CLASSES))]
assert actual == CLASSES, f'YOLO class order mismatch! got {actual}; expected {CLASSES}'
print('YOLO:', YOLO_PATH, '| classes=', len(actual), '| inference imgsz=', IMGSZ)


YOLO: /content/drive/MyDrive/YOLO_Courier_Backup_M/best.pt | classes= 38 | inference imgsz= 1024


## 6. Independent goal/via spatial grounding; V18 strict vs fallback Dijkstra

In [9]:
"""I Hate Phenikaa — V18 routing bridge.

STRICT: no target invention, no illegal action substituted.
FALLBACK: preserve all strict successes, only repair failure cases via
soft spatial grounding + exact V18 policy where possible + Dijkstra + legal step.

Action order: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT.
This module is independent of PyTorch/YOLO for deterministic unit tests.
"""
import heapq
import math
from collections import Counter, deque
from copy import deepcopy
from typing import Any

PLACES = ('canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports')
DIRECT = frozenset(('none','near','far','north','south','east','west'))
MAP = frozenset(('anchor_near','north_most','south_most','east_most','west_most'))
CONSTRAINED = (DIRECT - {'none'}) | MAP
DIRECTIONS = {0:(-1,0),1:(1,0),2:(0,-1),3:(0,1)}
DELTA_TO_ACTION = {v:k for k,v in DIRECTIONS.items()}
ACTION_NAMES = ('UP','DOWN','LEFT','RIGHT')
HEADING = {s:i for i,s in enumerate(ACTION_NAMES)}
RIGHT = {0:3,3:1,1:2,2:0}
LEFT = {v:k for k,v in RIGHT.items()}

class GroundingError(ValueError):
    pass


def rc(item):
    v = item.get('rc') if isinstance(item, dict) else item
    if not isinstance(v, (list,tuple)) or len(v) != 2:
        raise GroundingError('invalid rc '+repr(v))
    return (int(v[0]),int(v[1]))


def md(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])


def lms_of(scene, kind=None):
    out,seen=[],set()
    for lm in scene.get('landmarks',[]):
        if lm.get('type') not in PLACES:continue
        if kind is not None and lm['type']!=kind:continue
        pos=rc(lm); key=(lm['type'],pos)
        if key not in seen:
            seen.add(key);out.append({'type':lm['type'],'rc':list(pos)})
    return out


def _choose(cands, score, soft=False, origin=None):
    if not cands:raise GroundingError('no candidate landmarks')
    values=[float(score(x)) for x in cands]
    mn=min(values)
    tied=[x for x,v in zip(cands,values) if abs(v-mn)<1e-8]
    if len(tied)==1:return tied[0],False
    if not soft:raise GroundingError('spatial ambiguous '+str(len(tied))+' equal choices')
    origin=origin or (0,0)
    return min(tied,key=lambda lm:(md(rc(lm),origin),lm['type'],rc(lm))),True


def _geometry(cands,kind):
    if kind in ('north','north_most'):return lambda lm:rc(lm)[0]
    if kind in ('south','south_most'):return lambda lm:-rc(lm)[0]
    if kind in ('west','west_most'):return lambda lm:rc(lm)[1]
    if kind in ('east','east_most'):return lambda lm:-rc(lm)[1]
    raise GroundingError('unsupported geometry '+repr(kind))


def _proxy(scene, preferred=None, origin=None, exclude_types=()):
    lms=lms_of(scene)
    if preferred in PLACES:
        pref=[lm for lm in lms if lm['type']==preferred]
        if pref:lms=pref
    if exclude_types and len(lms)>1:
        filtered=[lm for lm in lms if lm['type'] not in exclude_types]
        if filtered:lms=filtered
    if not lms:raise GroundingError('no landmarks available for proxy')
    origin=origin or rc(scene['robot'])
    return min(lms,key=lambda lm:(md(rc(lm),origin),lm['type'],rc(lm)))


def ground_role(scene,raw,role,soft=False):
    """Return (type, spatial_ref, repaired, reason). Via absent is valid ONLY kind none."""
    assert role in ('goal','via')
    typ=raw.get(role)
    typ=None if typ in (None,'none','') else str(typ)
    ref=dict(raw.get(role+'_ref') or {})
    kind=ref.get('kind') or 'none'
    anc=ref.get('anchor')
    if anc=='none':anc=None
    origin=rc(scene['robot']) if scene.get('robot') and scene['robot'].get('rc') else (0,0)
    if kind not in DIRECT|MAP:raise GroundingError(role+': invalid kind '+repr(kind))
    if kind=='none' and typ is None:
        if role=='via':return None,None,False,'via_absent'
        if not soft:raise GroundingError('goal absent without a spatial kind')
        lm=_proxy(scene,origin=origin)
        return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'goal_proxy_no_label'
    if kind in MAP:
        if typ is not None and not soft:raise GroundingError(role+': map-dependent kind with named type in raw NLU')
        cands=lms_of(scene)
        if not cands:raise GroundingError(role+': no landmarks')
        repaired=False
        if kind=='anchor_near':
            anchors=lms_of(scene,anc)
            if not anc or not anchors:
                if not soft:raise GroundingError(role+': missing spatial anchor '+str(anc))
                lm=_proxy(scene,origin=origin,exclude_types=[anc] if anc else [])
                return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'anchor_near_missing_anchor_proxy'
            excluded={rc(a) for a in anchors}
            cands=[lm for lm in cands if rc(lm) not in excluded]
            if not cands:
                if not soft:raise GroundingError(role+': no candidate except anchor')
                lm=_proxy(scene,origin=origin)
                return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'anchor_near_no_other_proxy'
            score=lambda lm:min(md(rc(lm),rc(a)) for a in anchors)
        else:score=_geometry(cands,kind)
        lm,tied=_choose(cands,score,soft=soft,origin=origin)
        return lm['type'],{'kind':kind,'anchor':anc if kind=='anchor_near' else None,'rc':lm['rc']},tied,'map_tie_repaired' if tied else 'map_resolved'
    if typ is None:
        if not soft:raise GroundingError(role+': missing named type for '+kind)
        lm=_proxy(scene,origin=origin)
        return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'unnamed_direct_proxy'
    if typ not in PLACES:
        if not soft:raise GroundingError(role+': unknown type '+repr(typ))
        lm=_proxy(scene,origin=origin)
        return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'unknown_type_proxy'
    cands=lms_of(scene,typ)
    if not cands:
        if not soft:raise GroundingError(role+': YOLO missing '+typ)
        lm=_proxy(scene,origin=origin)
        return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'missing_named_type_proxy'
    if kind=='none':return typ,None,False,'named_unconstrained'
    if kind in ('near','far'):
        anchors=lms_of(scene,anc)
        if not anc or not anchors:
            if not soft:raise GroundingError(role+': anchor '+repr(anc)+' undetected')
            lm=_proxy(scene,preferred=typ,origin=origin)
            return lm['type'],{'kind':'none','anchor':None,'rc':lm['rc']},True,'named_missing_anchor_proxy'
        distances=lambda lm:min(md(rc(lm),rc(a)) for a in anchors)
        score=distances if kind=='near' else lambda lm:-distances(lm)
    else:score=_geometry(cands,kind)
    lm,tied=_choose(cands,score,soft=soft,origin=origin)
    return typ,{'kind':kind,'anchor':anc if kind in ('near','far') else None,'rc':lm['rc']},tied,'named_tie_repaired' if tied else 'named_spatial_resolved'


def ground_mission(scene,raw,soft=False):
    errors=[];details={}; mission={}
    if raw is None:raw={}
    for role in ('goal','via'):
        try:
            place,ref,repaired,reason=ground_role(scene,raw,role,soft)
            mission[role]=place;mission[role+'_ref']=ref
            details[role]={'reason':reason,'repaired':bool(repaired),'type':place,'rc':ref.get('rc') if ref else None}
        except (GroundingError,TypeError,KeyError,ValueError) as ex:
            errors.append(f'{role}: {str(ex)}')
            mission[role]=None; mission[role+'_ref']=None
    wanted_via=bool(raw.get('via_required', raw.get('via') not in (None,'none','') or (raw.get('via_ref') or {}).get('kind') in MAP))
    mission.update(urgent=bool(raw.get('urgent',False)),fragile=bool(raw.get('fragile',False)))
    mission['via_required']=wanted_via
    if wanted_via and not mission['via']:
        if soft and lms_of(scene):
            proxy=_proxy(scene,origin=rc(scene['robot']))
            mission['via']=proxy['type'];mission['via_ref']={'kind':'none','anchor':None,'rc':proxy['rc']}
            details['via']={'reason':'missing_required_via_proxy','repaired':True,'type':proxy['type'],'rc':proxy['rc']}
        else:errors.append('via: required but missing')
    if not mission['goal']:
        if soft and lms_of(scene):
            proxy=_proxy(scene,origin=rc(scene['robot']))
            mission['goal']=proxy['type'];mission['goal_ref']={'kind':'none','anchor':None,'rc':proxy['rc']}
            details['goal']={'reason':'missing_required_goal_proxy','repaired':True,'type':proxy['type'],'rc':proxy['rc']}
        else:errors.append('goal: required but missing')
    return mission,errors,details


def _nodes_from_scene(scene):
    points=set()
    for x in scene.get('nodes',[]):
        try:points.add(rc(x))
        except (ValueError,GroundingError):pass
    for x in scene.get('landmarks',[]):
        try:points.add(rc(x))
        except (ValueError,GroundingError):pass
    if scene.get('robot'):
        try:points.add(rc(scene['robot']))
        except (ValueError,GroundingError):pass
    for e in scene.get('edges',[]):
        try:points.add(rc(e['a']));points.add(rc(e['b']))
        except (KeyError,ValueError,GroundingError):pass
    return points


def normalize_scene(scene,mission,soft=False):
    """Robot V18-compatible scene; soft=True can drop malformed YOLO edges.
    No model-internal code changed.
    """
    raw=scene or {}
    robot=raw.get('robot') or {}
    if not robot or 'rc' not in robot:raise GroundingError('YOLO missing robot position')
    origin=rc(robot)
    heading=str(robot.get('heading') or '').upper()
    if heading not in HEADING:
        if not soft:raise GroundingError('YOLO missing robot heading')
        heading='UP'
    weather=str(raw.get('weather') or '').lower()
    if weather not in ('dry','rain'):
        if not soft:raise GroundingError('invalid weather '+repr(weather))
        weather='dry'
    out_edges=[];seen=set();dropped=0
    for e in raw.get('edges',[]):
        try:
            a,b=rc(e['a']),rc(e['b'])
            if (b[0]-a[0],b[1]-a[1]) not in DELTA_TO_ACTION:raise GroundingError('nonadjacent edge')
            status=str(e.get('status') or 'normal').lower()
            if status not in ('normal','crowded','covered','closed'):
                if not soft:raise GroundingError('unknown edge status '+status)
                status='normal'
            stairs=bool(e.get('stairs',False))
            ow=e.get('oneway_to');ow=rc(ow) if ow is not None else None
            if ow is not None and ow not in (a,b):raise GroundingError('bad oneway_to')
            key=(tuple(sorted((a,b))),status,stairs,ow)
            if key in seen:continue
            seen.add(key)
            out_edges.append({'a':list(a),'b':list(b),'status':status,'stairs':stairs,'oneway_to':list(ow) if ow else None})
        except (ValueError,TypeError,KeyError,GroundingError):
            dropped+=1
            if not soft:raise
    if not out_edges:raise GroundingError('YOLO graph missing usable edges')
    landmarks=lms_of(raw)
    if not landmarks and not soft:raise GroundingError('YOLO no landmarks')
    points=_nodes_from_scene(raw)
    points.update([origin]+[rc(e['a']) for e in out_edges]+[rc(e['b']) for e in out_edges])
    points.update(rc(x) for x in landmarks)
    grid=raw.get('grid') or {}
    rows=int(grid.get('rows',max(x[0] for x in points)+1))
    cols=int(grid.get('cols',max(x[1] for x in points)+1))
    if not (1<=rows<=9 and 1<=cols<=9) or any(not(0<=r<rows and 0<=c<cols) for r,c in points):
        raise GroundingError('unsupported/mismatched grid '+str((rows,cols)))
    for role in ('goal','via'):
        place=mission.get(role)
        if place is None:continue
        if place not in PLACES:raise GroundingError('bad '+role+' type '+repr(place))
        ref=mission.get(role+'_ref') or {}
        if ref.get('rc') is not None:
            pos=rc(ref)
            if (place,pos) not in {(x['type'],rc(x)) for x in landmarks}:
                raise GroundingError(role+': pinned landmark not in graph')
        elif not any(x['type']==place for x in landmarks):
            raise GroundingError(role+': place not detected '+place)
        if not soft and ref.get('kind') in CONSTRAINED and ref.get('rc') is None:
            raise GroundingError(role+': spatial unresolved')
    if not soft and not mission.get('goal'):raise GroundingError('goal unresolved')
    if not soft and mission.get('via_required') and not mission.get('via'):
        raise GroundingError('required via unresolved')
    result={
        'scene_id':str(raw.get('scene_id','yolo-scene')),
        'robot':{'rc':list(origin),'heading':heading},
        'nodes':[{'rc':list(pos)} for pos in sorted(points)],
        'landmarks':landmarks,'edges':out_edges,
        'weather':weather,'mission':{
            'goal':mission.get('goal'), 'via':mission.get('via'),
            'goal_ref':mission.get('goal_ref'), 'via_ref':mission.get('via_ref'),
            'urgent':bool(mission.get('urgent',False)),'fragile':bool(mission.get('fragile',False))
        },
        'grid':{'rows':rows,'cols':cols},
        'style':str(raw.get('style','unknown')) if str(raw.get('style','unknown')) in ('classic','night','sketch','print','clean','unknown') else 'unknown',
    }
    return result,dropped


def allowed_edges(scene,rid):
    adj={}
    for e in scene.get('edges',[]):
        if e.get('status')=='closed':continue
        if e.get('stairs') and rid!=4:continue
        try:a,b=rc(e['a']),rc(e['b'])
        except (ValueError,TypeError,GroundingError):continue
        ow=e.get('oneway_to');ow=rc(ow) if ow is not None else None
        for u,v in ((a,b),(b,a)):
            if ow is not None and ow!=v:continue
            step=(v[0]-u[0],v[1]-u[1])
            if step in DELTA_TO_ACTION:
                adj.setdefault(u,[]).append((v,DELTA_TO_ACTION[step],e))
    return adj


def legal_actions(scene,rid):
    try:start=rc(scene['robot'])
    except (KeyError,ValueError,GroundingError):return set()
    return {d for _,d,_ in allowed_edges(scene,rid).get(start,[])}


def target_positions(scene,role):
    m=scene.get('mission') or {}
    typ=m.get(role)
    if not typ:return set()
    cands=lms_of(scene,typ)
    ref=m.get(role+'_ref') or {}
    if ref.get('rc') is not None:return {rc(ref)}
    return {rc(x) for x in cands}


def route_reachable(scene,rid):
    """Directed connectivity only, not an alternative primary robot policy."""
    start=rc(scene['robot']);goal=target_positions(scene,'goal');via=target_positions(scene,'via')
    if not goal or (scene['mission'].get('via') and not via):return False
    adj=allowed_edges(scene,rid)
    todo=deque([(start,False if via else True)]);seen=set(todo)
    while todo:
        node,done=todo.popleft()
        done=done or node in via
        if done and node in goal:return True
        for nxt,_,_ in adj.get(node,[]):
            state=(nxt,done or nxt in via)
            if state not in seen:seen.add(state);todo.append(state)
    return False


def _edge_cost(e,rid,weather,urgent,fragile):
    # *Fallback heuristic* only. Not claimed to be learned hidden V18 cost.
    stat=e.get('status','normal')
    cost={'normal':1.0,'crowded':2.6,'covered':1.2,'closed':math.inf}.get(stat,1.0)
    if stat=='covered' and weather=='rain':cost=.9
    if e.get('stairs'):
        cost+=0.5+(0.9 if weather=='rain' else 0)+(0.6 if fragile else 0)
    if stat=='crowded' and fragile:cost+=.5
    if urgent:cost=max(.2,cost*.95)
    return cost


def dijkstra_fallback(scene,rid):
    """Heading-aware, directed, via-then-goal Dijkstra, returns action or None.
    Used ONLY after strict V18 cannot provide a trusted action.
    """
    start=rc(scene['robot'])
    heading=HEADING.get(str(scene['robot'].get('heading','UP')).upper(),0)
    goals=target_positions(scene,'goal')
    vias=target_positions(scene,'via')
    if not goals:return None
    if scene['mission'].get('via') and not vias:return None
    hasvia=bool(vias)
    phase=0 if hasvia and start not in vias else 1
    adj=allowed_edges(scene,rid)
    if not adj.get(start):return None
    weather=scene.get('weather','dry')
    urgent=scene['mission'].get('urgent',False)
    fragile=scene['mission'].get('fragile',False)
    counter=0
    heap=[(0.,0,0,counter,start,heading,phase,-1)]
    best={(start,heading,phase):-0.00000001}
    while heap:
        cost,turns,steps,_,pos,hd,phase,first=heapq.heappop(heap)
        if cost>best.get((pos,hd,phase),math.inf)+1e-7:continue
        if phase==1 and pos in goals and first!=-1:return first
        for nxt,d,e in adj.get(pos,[]):
            nphase=1 if phase or nxt in vias else 0
            turn=0 if d==hd else (2 if d==RIGHT[RIGHT[hd]] else 1)
            w=_edge_cost(e,rid,weather,urgent,fragile)+.12*turn
            cand=cost+w
            state=(nxt,d,nphase)
            if cand+1e-9<best.get(state,math.inf):
                best[state]=cand;counter+=1
                heapq.heappush(heap,(cand,turns+turn,steps+1,counter,nxt,d,nphase,d if first==-1 else first))
    return None


def local_legal_fallback(scene,rid):
    """Last resort when no goal/anchor/landmark/path exists; never traverse closed/oneway/stairs-illegal edge."""
    start=rc(scene['robot']);adj=allowed_edges(scene,rid);neighbors=adj.get(start,[])
    if not neighbors:return None
    landmarks=[rc(x) for x in lms_of(scene)]
    hd=HEADING.get(str(scene['robot'].get('heading','UP')).upper(),0)
    order=(hd,RIGHT[hd],LEFT[hd],RIGHT[RIGHT[hd]])
    def score(item):
        n,action,e=item
        d=min((md(n,x) for x in landmarks),default=0)
        return (d,_edge_cost(e,rid,scene.get('weather','dry'),False,False),order.index(action),action)
    return min(neighbors,key=score)[1]


def repair_scene(raw,mission_raw):
    """Construct fallback scene; do not mutate strict/Yolo source."""
    clone=deepcopy(raw)
    # No robot position => cannot produce a *legal* movement.
    if not clone.get('robot') or 'rc' not in clone['robot']:
        raise GroundingError('no robot position: no fallback possible')
    mission,errors,details=ground_mission(clone,mission_raw,soft=True)
    # If there are zero YOLO landmarks, all goals remain None. Local movement still possible.
    if not mission.get('goal'):
        mission['goal']=None;mission['goal_ref']=None
    norm,dropped=normalize_scene(clone,mission,soft=True)
    return norm,errors,details,dropped


def policy_strict_scene(raw,raw_mission):
    """Ground + validate (NO fallback). Return normalized scene or reason."""
    try:
        mission,errors,details=ground_mission(raw,raw_mission,soft=False)
        if errors:raise GroundingError('; '.join(errors))
        norm,dropped=normalize_scene(raw,mission,soft=False)
        return norm,None,details
    except (GroundingError,ValueError,TypeError,KeyError) as e:
        return None,f'{type(e).__name__}: {e}',{}


def route_results(raw_scenes,raw_missions,policy_predict,batch_size=48,debug=False):
    """Run same inputs through STRICT and FALLBACK. Strict results frozen independently.
    Policy must accept list of normalized scenes and return [N,10] integer actions.
    """
    if len(raw_scenes)!=len(raw_missions):raise ValueError('different scene/mission lengths')
    n=len(raw_scenes)
    strict=[[-1]*10 for _ in range(n)]
    fallback=[[-1]*10 for _ in range(n)]
    why=[[] for _ in range(n)]; repairs=[{} for _ in range(n)]
    strict_norm=[None]*n
    good=[];ids=[]
    for i,(raw,mission) in enumerate(zip(raw_scenes,raw_missions)):
        sc,err,detail=policy_strict_scene(raw,mission)
        if err:
            why[i].append('strict_ground: '+err)
            continue
        strict_norm[i]=sc; good.append(sc);ids.append(i)
    # Run V18 once per batch, not once per robot. Fail only affected batches on inference exceptions.
    for start in range(0,len(good),batch_size):
        batch=good[start:start+batch_size];indexes=ids[start:start+batch_size]
        try:
            pred=policy_predict(batch)
            import numpy as np
            pred=np.asarray(pred)
            if pred.shape!=(len(batch),10):raise RuntimeError('V18 wrong output shape '+str(pred.shape))
        except KeyboardInterrupt:raise
        except Exception as e:
            for i in indexes:why[i].append('V18 execution: '+type(e).__name__+': '+str(e)[:300])
            continue
        for j,i in enumerate(indexes):
            sc=batch[j]
            for rid in range(10):
                a=int(pred[j,rid]); legal=legal_actions(sc,rid)
                if a not in legal:
                    why[i].append('R'+str(rid)+' illegal_or_minus1: '+str(a))
                elif rid != 9 and not route_reachable(sc,rid):
                    why[i].append('R'+str(rid)+' no_reachable_route')
                else:strict[i][rid]=a
    # Fallback only for strict=-1. Never change any successful V18 decision.
    # Repair all scenes first, then call V18 on repaired scenes IN BATCHES.
    repaired_scenes={};repair_ids=[]
    for i in range(n):
        fallback[i]=strict[i][:]
        if all(a>=0 for a in strict[i]):continue
        try:
            sc,errs,detail,dropped=repair_scene(raw_scenes[i],raw_missions[i])
            repairs[i]={'details':detail,'soft_errors':errs,'dropped_bad_edges':dropped}
            repaired_scenes[i]=sc
            if sc.get('mission',{}).get('goal'):repair_ids.append(i)
        except Exception as e:
            why[i].append('fallback_invalid_graph: '+str(e))

    for begin in range(0,len(repair_ids),batch_size):
        indexes=repair_ids[begin:begin+batch_size]
        batch=[repaired_scenes[i] for i in indexes]
        try:
            import numpy as np
            actions=np.asarray(policy_predict(batch))
            if actions.shape!=(len(batch),10):
                raise RuntimeError('V18 repaired output wrong shape '+str(actions.shape))
        except KeyboardInterrupt:raise
        except Exception as e:
            for i in indexes:why[i].append('fallback_V18_exception '+type(e).__name__+': '+str(e)[:160])
            continue
        for j,i in enumerate(indexes):
            sc=batch[j]
            for rid in range(10):
                if strict[i][rid]!=-1:continue
                action=int(actions[j,rid])
                if action in legal_actions(sc,rid) and (rid==9 or route_reachable(sc,rid)):
                    fallback[i][rid]=action
                    why[i].append('R'+str(rid)+' repaired_then_V18')

    # Only remaining failures invoke Dijkstra or a legal-neighbor fallback.
    for i,sc in repaired_scenes.items():
        for rid in range(10):
            if fallback[i][rid]>=0:continue
            action=dijkstra_fallback(sc,rid) if sc.get('mission',{}).get('goal') else None
            if action is not None and action in legal_actions(sc,rid):
                fallback[i][rid]=action;why[i].append('R'+str(rid)+' fallback_dijkstra')
            else:
                action=local_legal_fallback(sc,rid)
                if action is not None:
                    fallback[i][rid]=action;why[i].append('R'+str(rid)+' fallback_local_legal')
                else:why[i].append('R'+str(rid)+' physically_stuck')
    report={
        'scenes':n,'strict_minus1':sum(a<0 for x in strict for a in x),
        'fallback_minus1':sum(a<0 for x in fallback for a in x),
        'strict_scenes_with_minus1':sum(any(a<0 for a in row) for row in strict),
        'fallback_scenes_with_minus1':sum(any(a<0 for a in row) for row in fallback),
        'recovered':sum(a<0 and b>=0 for x,y in zip(strict,fallback) for a,b in zip(x,y)),
        'modified_good_actions':sum(a>=0 and b!=a for x,y in zip(strict,fallback) for a,b in zip(x,y)),
        'reasons':Counter(w.split(':',1)[0] for ws in why for w in ws).most_common(30),
    }
    assert report['modified_good_actions']==0
    return {'strict':strict,'fallback':fallback,'reasons':why,'repairs':repairs,'report':report}


# =========== V18 empty-primary isolated routing override ===========
"""Harden V18 against empty primary candidate arrays without changing successful policies.
Batch isolation is scene-disjoint. There is no broad fallback for valid strict decisions.
"""
from collections import defaultdict

def _primary_shapes(scene):
    """Per-scene debug data printed on errors, not a substitute for the real Q model."""
    import numpy as np
    goal=np.asarray(sorted(target_positions(scene,'goal')),dtype=np.int16).reshape(-1,2)
    via=np.asarray(sorted(target_positions(scene,'via')),dtype=np.int16).reshape(-1,2)
    start=rc(scene['robot'])
    legal={f'R{rid}':sorted(legal_actions(scene,rid)) for rid in range(10)}
    reachable={f'R{rid}':(rid==9 or route_reachable(scene,rid)) for rid in range(10)}
    return {'scene_id':str(scene.get('scene_id','?')),'goal_coords_shape':list(goal.shape),
            'via_coords_shape':list(via.shape),'via_required':bool(scene['mission'].get('via')),
            'legal_actions':legal,'reachable':reachable,'start':list(start),
            'goal_kind':(scene['mission'].get('goal_ref') or {}).get('kind','none'),
            'via_kind':(scene['mission'].get('via_ref') or {}).get('kind','none')}


def _unsafe_primary(diag):
    """No model call on rows known to have an empty goal or an impossible start/path.
    R9 is greedy but shared V18 runtime still evaluates all robots, so isolate unsafe rows.
    """
    if diag['goal_coords_shape'][0]==0:return 'empty_goals'
    if diag['via_required'] and diag['via_coords_shape'][0]==0:return 'empty_required_via'
    if any(not diag['legal_actions'][f'R{k}'] for k in range(10)):return 'empty_legal_actions'
    if any(not diag['reachable'][f'R{k}'] for k in range(9)):return 'unreachable_primary'
    return None


def _run_v18_isolated(rows, ids, policy_predict, batch_size, reasons, stage, debug=False):
    """Run all good scenes in batch, bisect only batches raising exceptions.
    Returns dict(index -> prediction[10]); preserves successes while isolating bad rows.
    """
    import numpy as np
    import json
    result={}
    def attempt(chunk, indices, depth=0):
        if not indices:return
        try:
            with np.errstate(all='ignore'):
                arr=np.asarray(policy_predict(chunk))
            if arr.shape != (len(chunk),10):
                raise ValueError(f'bad V18 shape {arr.shape}; expected {(len(chunk),10)}')
            if not np.issubdtype(arr.dtype,np.number):raise TypeError(f'non-numeric policy predictions {arr.dtype}')
            if not np.all(np.isfinite(arr)):raise ValueError(f'nonfinite policy predictions shape {arr.shape}')
            for i,row in zip(indices,arr):result[i]=[int(a) for a in row]
        except KeyboardInterrupt:raise
        except Exception as exc:
            if len(indices)>1:
                mid=len(indices)//2
                attempt(chunk[:mid],indices[:mid],depth+1)
                attempt(chunk[mid:],indices[mid:],depth+1)
            else:
                i=indices[0]
                try:diag=_primary_shapes(chunk[0])
                except Exception as de:diag={'diagnostic_error':str(de)}
                label='empty_primary' if ('empty' in str(exc).lower() or 'zero-size array' in str(exc).lower() or 'argmin' in str(exc).lower()) else 'policy_exception'
                reasons[i].append(f'{stage}_{label}: {type(exc).__name__}: {str(exc)[:260]}')
                reasons[i].append('primary_arrays: '+json.dumps(diag,ensure_ascii=False,sort_keys=True))
    for start in range(0,len(ids),max(1,batch_size)):
        take=ids[start:start+batch_size]
        attempt([rows[i] for i in take],take)
    return result


def route_results(raw_scenes,raw_missions,policy_predict,batch_size=48,debug=False):
    """Robust STRICT and FALLBACK V18. Never changes valid strict predictions.
    Strict output may contain -1; fallback tries V18 on repaired scenes then Dijkstra/local.
    """
    import json
    from collections import Counter
    if len(raw_scenes)!=len(raw_missions):raise ValueError('different scene/mission lengths')
    n=len(raw_scenes)
    strict=[[-1]*10 for _ in range(n)]
    fallback=[[-1]*10 for _ in range(n)]
    why=[[] for _ in range(n)];repairs=[{} for _ in range(n)]
    normalized={};candidate_ids=[];prefilter=Counter()
    for i,(raw,rawmission) in enumerate(zip(raw_scenes,raw_missions)):
        sc,err,detail=policy_strict_scene(raw,rawmission)
        if err:
            why[i].append('strict_ground: '+err)
            prefilter['strict_ground']+=1
            continue
        normalized[i]=sc
        try:
            diag=_primary_shapes(sc)
            issue=_unsafe_primary(diag)
        except KeyboardInterrupt:raise
        except Exception as exc:
            why[i].append(f'strict_primary_diagnostic: {type(exc).__name__}: {exc}')
            prefilter['diagnostic_exception']+=1
            continue
        if issue:
            why[i].append('strict_empty_or_unreachable_primary: '+issue+' | '+json.dumps(diag,ensure_ascii=False))
            prefilter[issue]+=1
            continue
        candidate_ids.append(i)
    preds=_run_v18_isolated(normalized,candidate_ids,policy_predict,batch_size,why,'strict',debug)
    for i,p in preds.items():
        sc=normalized[i]
        for rid,a in enumerate(p):
            try:
                if a not in legal_actions(sc,rid):
                    why[i].append(f'R{rid} strict_invalid_action: {a}')
                elif rid!=9 and not route_reachable(sc,rid):
                    why[i].append(f'R{rid} strict_unreachable')
                else:strict[i][rid]=a
            except KeyboardInterrupt:raise
            except Exception as exc:why[i].append(f'R{rid} strict_validation_error: {type(exc).__name__}: {str(exc)[:100]}')
    # Fallback: if primary V18 fails, repair scene and apply policy only on safe repaired rows.
    repaired={};candidate_repaired=[]
    for i in range(n):
        fallback[i]=strict[i][:]
        if all(a>=0 for a in strict[i]):continue
        try:
            sc,errs,detail,dropped=repair_scene(raw_scenes[i],raw_missions[i])
            repaired[i]=sc
            repairs[i]={'details':detail,'soft_errors':errs,'dropped_bad_edges':dropped}
            diag=_primary_shapes(sc)
            issue=_unsafe_primary(diag)
            if issue:
                why[i].append('repaired_primary_unavailable: '+issue+' | '+json.dumps(diag,ensure_ascii=False))
            elif i in normalized and sc == normalized[i] and any(
                    x.startswith(('strict_empty_primary:', 'strict_policy_exception:')) for x in why[i]):
                # Identical input already failed; avoid loading all frozen models a second time.
                why[i].append('repaired_V18_skipped_identical_failed_input')
            else:candidate_repaired.append(i)
        except KeyboardInterrupt:raise
        except Exception as exc:
            why[i].append(f'fallback_invalid_graph: {type(exc).__name__}: {str(exc)[:180]}')
    predicted_repaired=_run_v18_isolated(repaired,candidate_repaired,policy_predict,batch_size,why,'repaired',debug)
    for i,p in predicted_repaired.items():
        sc=repaired[i]
        for rid,a in enumerate(p):
            if strict[i][rid]>=0:continue
            if a in legal_actions(sc,rid) and (rid==9 or route_reachable(sc,rid)):
                fallback[i][rid]=a
                why[i].append(f'R{rid} repaired_then_V18')
    for i,sc in repaired.items():
        for rid in range(10):
            if fallback[i][rid]>=0:continue
            action=None
            if sc.get('mission',{}).get('goal'):
                try:action=dijkstra_fallback(sc,rid)
                except KeyboardInterrupt:raise
                except Exception as exc:why[i].append(f'R{rid} dijkstra_exception: {type(exc).__name__}: {str(exc)[:160]}')
            if action is not None and action in legal_actions(sc,rid):
                fallback[i][rid]=action;why[i].append(f'R{rid} fallback_dijkstra')
                continue
            try:action=local_legal_fallback(sc,rid)
            except KeyboardInterrupt:raise
            except Exception as exc:
                why[i].append(f'R{rid} local_exception: {type(exc).__name__}: {str(exc)[:160]}')
                action=None
            if action is not None and action in legal_actions(sc,rid):
                fallback[i][rid]=action;why[i].append(f'R{rid} fallback_local_legal')
            else:why[i].append(f'R{rid} physically_stuck')
    report={
        'scenes':n,
        'strict_minus1':sum(a<0 for line in strict for a in line),
        'fallback_minus1':sum(a<0 for line in fallback for a in line),
        'strict_scenes_with_minus1':sum(any(a<0 for a in line) for line in strict),
        'fallback_scenes_with_minus1':sum(any(a<0 for a in line) for line in fallback),
        'recovered':sum(a<0 and b>=0 for l,m in zip(strict,fallback) for a,b in zip(l,m)),
        'modified_good_actions':sum(a>=0 and b!=a for l,m in zip(strict,fallback) for a,b in zip(l,m)),
        'preflight':dict(prefilter),
        'reasons':Counter(reason.split(':',1)[0] for lines in why for reason in lines).most_common(35),
    }
    assert report['modified_good_actions']==0,'fallback modified valid strict action'
    return {'strict':strict,'fallback':fallback,'reasons':why,'repairs':repairs,'report':report}



## 6b. New: full-route tie break and via-only recovery for all 10 robots

Do not choose equal landmarks solely by Manhattan distance from the robot.
The routing patch replaces the old `route_results` only; frozen V18 policies stay unchanged.
STRICT can resolve a valid spatial tie using full-route Dijkstra; FALLBACK additionally handles missing goal, missing anchor, incomplete paths and legal last-resort steps.


In [10]:
"""V18 FULL-ROUTE TIE RESOLVER + VIA-FIRST FALLBACK.

This module is inserted after the original V18 bridge cell so it reuses its
validated grid/edge/action helpers and trained V18 policy. No new robot weights.

STRICT: normal nonambiguous scenes use exact frozen V18 policy; when a named
spatial constraint has multiple equally optimal landmarks, resolve only that
semantic tie by heading-aware complete-route search on legally traversable edges.
Missing goal/anchor remains -1 in STRICT.

FALLBACK: keep every strict action. On failure use only valid detected landmarks
and directed graph. Joint goal+via => total route; goal missing/inaccessible but
via reachable => route to via only. No target => legal local step or -1.
"""
import heapq
import math
import json
from collections import Counter
from copy import deepcopy


def _frc(x):
    return rc(x)


def _spatial_options(scene, raw, role, allow_repair=False):
    """Return all spatially optimal candidate landmark objects, not an early tie winner.

    `raw` is decoded pre-map NLU. When no named landmark or anchor is detected,
    allow_repair uses observed fallback candidates only (never hallucinated rc).
    """
    if raw is None: raw={}
    typ=raw.get(role)
    typ=None if typ in (None,'none','') else str(typ)
    ref=raw.get(role+'_ref') or {}
    kind=ref.get('kind') or 'none'
    anchor=ref.get('anchor')
    anchor=None if anchor in (None,'none','') else str(anchor)
    if kind not in DIRECT | MAP:
        raise GroundingError(f'{role}: invalid spatial kind {kind!r}')
    all_lm=lms_of(scene)
    required=(role=='goal' or bool(raw.get('via_required', typ is not None or kind in MAP)))
    if role=='via' and not required and kind=='none' and typ is None:
        return [],'via_absent',False
    has_via=bool(raw.get('via_required',
                 raw.get('via') not in (None,'none','') or
                 (raw.get('via_ref') or {}).get('kind') in MAP))
    if role=='goal' and typ is None and kind=='none' and has_via:
        return [],'goal_absent_route_via_only',True

    if kind in MAP:
        if typ is not None and not allow_repair:
            raise GroundingError(f'{role}: map spatial kind with named type')
        pool=lms_of(scene,typ) if typ in PLACES and allow_repair and lms_of(scene,typ) else all_lm
    elif typ in PLACES:
        pool=lms_of(scene,typ)
        if not pool and allow_repair:
            # A named GOAL missing from YOLO is not evidence that another
            # landmark is the correct goal. If VIA is known, go VIA-only.
            # If VIA itself is missing, recover a possible waypoint from
            # observed landmarks, then search for the named GOAL.
            pool=[] if role=='goal' and has_via else all_lm
    elif allow_repair:
        pool=all_lm
    else:
        raise GroundingError(f'{role}: named type missing for {kind}')

    if not pool:
        if allow_repair:
            return [],f'{role}: no detected landmarks',True
        raise GroundingError(f'{role}: no landmark candidates, type={typ}, kind={kind}')
    repaired=bool((typ not in PLACES and kind not in MAP) or (typ in PLACES and not lms_of(scene,typ)))
    why=f'{role}: candidates={len(pool)} kind={kind}'

    if kind=='anchor_near' or kind in ('near','far'):
        anchors=lms_of(scene,anchor) if anchor in PLACES else []
        if not anchors:
            if not allow_repair:
                raise GroundingError(f'{role}: anchor not found {anchor!r}')
            if role=='goal' and kind=='anchor_near' and has_via:
                return [],why+' missing_anchor_route_via_only',True
            # Anchor missing: preserve the candidate pool. Let graph costs
            # select, rather than choosing Manhattan-nearest robot prematurely.
            return pool,why+' missing_anchor',True
        if kind=='anchor_near':
            positions={rc(a) for a in anchors}
            filtered=[lm for lm in pool if rc(lm) not in positions]
            if not filtered:
                if not allow_repair:
                    raise GroundingError(f'{role}: no candidate outside anchor')
                return pool,why+' only_anchor_left',True
            pool=filtered
        ds=[min(md(rc(lm),rc(a)) for a in anchors) for lm in pool]
        best=min(ds) if kind in ('near','anchor_near') else max(ds)
        pool=[lm for lm,d in zip(pool,ds) if d==best]
        why+=f' spatial_dist={best}'
    elif kind in ('north','north_most','south','south_most','east','east_most','west','west_most'):
        fn=_geometry(pool,kind)
        scores=[fn(lm) for lm in pool]
        m=min(scores)
        pool=[lm for lm,v in zip(pool,scores) if v==m]
        why+=f' spatial_axis={m}'
    elif kind!='none':
        raise GroundingError(f'{role}: unhandled kind {kind!r}')
    return sorted(pool,key=lambda lm:(rc(lm),lm['type'])),why,repaired


def _prepared_graph(scene,rid):
    # Honor each robot's exact legal graph according to V18 bridge rules.
    adj=allowed_edges(scene,rid)
    for pos in adj:
        adj[pos].sort(key=lambda z:(z[1],z[0]))
    return adj


def _relative_rank(heading,action):
    order=(heading,LEFT[heading],RIGHT[heading],RIGHT[RIGHT[heading]])
    return order.index(action)


def _turn_units(old_heading, action):
    if action==old_heading: return 0
    if action==RIGHT[RIGHT[old_heading]]:return 2
    return 1


def _full_route_option(scene,rid,via_options,goal_options,mode='strict'):
    """Shortest route over full route state (node, heading, via-done).

    Candidate sets are the Manhattan/axis-optimal landmark types/locations.
    Each candidate waypoint is evaluated jointly with *all* goal candidates.
    State keeps robot arrival heading; one-way and stairs remain enforced.
    Dijkstra cost here is fallback heuristic, not claimed as exact robot oracle.
    Returns metadata including chosen endpoint and initial action.
    """
    robot=scene.get('robot') or {}
    if 'rc' not in robot:return None
    start=rc(robot)
    heading=HEADING.get(str(robot.get('heading','UP')).upper(),0)
    vias={rc(v) for v in via_options}
    goals={rc(g) for g in goal_options}
    if not vias and not goals:return None
    if not goals and vias:kind='via_only'
    elif goals and vias:kind='via_goal'
    else:kind='goal_only'
    adj=_prepared_graph(scene,rid)
    if not adj.get(start):return None
    urgent=bool(scene.get('mission',{}).get('urgent',False))
    fragile=bool(scene.get('mission',{}).get('fragile',False))
    weather=scene.get('weather','dry')
    phase0=0 if vias and start not in vias else 1
    # When only VIA is requested and robot already at waypoint, no first step
    # is necessary; route result None and last-resort legal fallback may act.
    if kind=='via_only' and start in vias:return None
    best={}
    # (cost,turns,steps,first-relative-priority,first-action)
    initial=(0.,0,0,4,4)
    state0=(start,heading,phase0)
    best[state0]=initial
    q=[]; seq=0
    heapq.heappush(q,(*initial,seq,start,heading,phase0,None,None))
    while q:
        cost,turns,steps,pref,first,_,pos,hd,phase,chosen_via,chosen_goal=heapq.heappop(q)
        state=(pos,hd,phase)
        k=(cost,turns,steps,pref,first)
        if best.get(state)!=k:continue
        finished=(phase==1 and pos in goals) if kind!='via_only' else pos in vias
        if finished and first!=4:
            return {'action':first,'objective':kind,'cost':cost,'turn_units':turns,
                    'hops':steps,'via':chosen_via,'goal':pos if kind!='via_only' else None,
                    'robot_id':rid,'mode':mode}
        for nxt,d,edge in adj.get(pos,[]):
            units=_turn_units(hd,d)
            w=_edge_cost(edge,rid,weather,urgent,fragile)+.12*units
            if not math.isfinite(w) or w<0:continue
            nxt_phase=1 if phase or nxt in vias else 0
            first_a=d if first==4 else first
            first_pref=_relative_rank(heading,first_a)
            # Explicit fixed precision helps deterministic ties across Python versions.
            nk=(round(cost+w,9),turns+units,steps+1,first_pref,first_a)
            nst=(nxt,d,nxt_phase)
            if nst not in best or nk < best[nst]:
                best[nst]=nk
                seq+=1
                cv=chosen_via if chosen_via is not None else (nxt if nxt in vias else None)
                heapq.heappush(q,(*nk,seq,nxt,d,nxt_phase,cv,
                                    nxt if kind!='via_only' and nxt in goals else chosen_goal))
    return None


def _legal_local_recovery(scene,rid,prefer=None):
    """First try a useful legal neighbor; never invent a movement outside graph."""
    adj=_prepared_graph(scene,rid)
    robot=scene.get('robot') or {}
    if 'rc' not in robot:return None
    here=rc(robot)
    neighbors=adj.get(here,[])
    if not neighbors:return None
    anchors=[rc(x) for x in (prefer or [])]
    if not anchors:
        anchors=[rc(x) for x in lms_of(scene)]
    h=HEADING.get(str(robot.get('heading','UP')).upper(),0)
    def key(item):
        dest,action,edge=item
        distance=min((md(dest,t) for t in anchors),default=0)
        return (distance, _edge_cost(edge,rid,scene.get('weather','dry'),
                                      scene.get('mission',{}).get('urgent',False),
                                      scene.get('mission',{}).get('fragile',False)),
                _relative_rank(h,action),action)
    return min(neighbors,key=key)[1]


def _scene_for_options(scene,raw_mission,via_options,goal_options,soft):
    """Convert raw detected graph to normalized V18 scene, without lossy tie pick.

    Per-robot routing takes endpoints from via_options/goal_options, NOT the
    placeholder pinned references used solely to validate normalized graph.
    """
    mission={
        'goal':goal_options[0]['type'] if goal_options else None,
        'via':via_options[0]['type'] if via_options else None,
        'goal_ref':({'kind':'none','anchor':None,'rc':goal_options[0]['rc']} if goal_options else None),
        'via_ref':({'kind':'none','anchor':None,'rc':via_options[0]['rc']} if via_options else None),
        'urgent':bool(raw_mission.get('urgent',False)),
        'fragile':bool(raw_mission.get('fragile',False)),
        'via_required':bool(raw_mission.get('via_required',False)),
    }
    norm,dropped=normalize_scene(scene,mission,soft=soft)
    # The candidate list is for route optimization; originals remain unchanged.
    return norm,dropped


def _recover_options(raw_scene,raw_mission,soft):
    info={}; options={}
    for role in ('goal','via'):
        try:
            cands,how,repaired=_spatial_options(raw_scene,raw_mission,role,allow_repair=soft)
        except (GroundingError,TypeError,KeyError,ValueError) as ex:
            if not soft:raise
            cands,how,repaired=[],f'{role} unresolvable {type(ex).__name__}: {ex}',True
        options[role]=cands
        info[role]={'candidates':len(cands),'reason':how,'repaired':repaired,
                    'coordinates':[list(rc(x)) for x in cands]}
    return options,info


def _is_pure_spatial_tie(msg):
    return 'spatial ambiguous' in str(msg)


def route_results(raw_scenes,raw_missions,policy_predict,batch_size=128,debug=False):
    """Two independent outputs: STRICT (V18 or valid full-route tie), FALLBACK.

    Goal missing but a via exists => via-only fallback, NOT arbitrary detour.
    Route comparison always includes full via->goal when reachable.
    All 10 robots are processed independently using robot-specific legal graph.
    """
    import numpy as np
    if len(raw_scenes)!=len(raw_missions):raise ValueError('scene/mission lengths differ')
    n=len(raw_scenes)
    strict=[[-1]*10 for _ in range(n)]; fallback=[[-1]*10 for _ in range(n)]
    reasons=[[] for _ in range(n)]; repairs=[{} for _ in range(n)]
    normalized={}; norm_ids=[]; strict_tie={}; unresolved=[]
    for i,(raw,m) in enumerate(zip(raw_scenes,raw_missions)):
        m=m or {}
        sc,err,details=policy_strict_scene(raw,m)
        if err:
            reasons[i].append('strict_ground: '+err)
            if _is_pure_spatial_tie(err):
                # Resolve real spatial equality using *full* graph route;
                # do not fabricate missing anchors or landmarks in STRICT.
                try:
                    opts,inf=_recover_options(raw,m,soft=False)
                    if not opts['goal'] or (m.get('via_required') and not opts['via']):
                        raise GroundingError('missing goal or required via')
                    strict_scene,_=_scene_for_options(raw,m,opts['via'],opts['goal'],soft=False)
                    strict_tie[i]=(strict_scene,opts,inf)
                    reasons[i].append('strict_spatial_tie_full_route')
                except (GroundingError,KeyError,TypeError,ValueError) as ex:
                    reasons[i].append(f'strict_tie_unresolved: {ex}')
            continue
        normalized[i]=sc
        try:
            diag=_primary_shapes(sc)
            issue=_unsafe_primary(diag)
        except (KeyboardInterrupt,SystemExit):raise
        except Exception as e:
            reasons[i].append(f'strict_primary_diagnostic: {type(e).__name__}: {e}')
            continue
        if issue:
            reasons[i].append('strict_primary_unavailable: '+issue)
            continue
        norm_ids.append(i)

    # Frozen V18 remains the only primary policy for normal reachable scenes.
    preds=_run_v18_isolated(normalized,norm_ids,policy_predict,batch_size,reasons,'strict',debug)
    for i,p in preds.items():
        sc=normalized[i]
        for rid,action in enumerate(p):
            if action not in legal_actions(sc,rid):
                reasons[i].append(f'R{rid} invalid_primary_action: {action}')
            elif rid!=9 and not route_reachable(sc,rid):
                reasons[i].append(f'R{rid} primary_no_full_route')
            else:strict[i][rid]=int(action)

    # Strict tie resolution: graph-aware, 10 independent routes, no proxy.
    for i,(scene,opts,inf) in strict_tie.items():
        for rid in range(10):
            route=_full_route_option(scene,rid,opts['via'],opts['goal'],mode='strict_tie')
            if route and route['action'] in legal_actions(scene,rid):
                strict[i][rid]=route['action']
                reasons[i].append(f'R{rid} strict_tie_total_route {route["objective"]} cost={route["cost"]}')

    # No change to valid strict actions. Failed scenes are handled individually.
    for i,(raw,m) in enumerate(zip(raw_scenes,raw_missions)):
        fallback[i]=strict[i][:]
        if all(a>=0 for a in strict[i]):continue
        m=m or {}
        try:
            opts,inf=_recover_options(raw,m,soft=True)
            sc,dropped=_scene_for_options(raw,m,opts['via'],opts['goal'],soft=True)
            repairs[i]={'spatial':inf,'dropped_edges':dropped}
        except (KeyboardInterrupt,SystemExit):raise
        except Exception as ex:
            reasons[i].append(f'fallback_scene_unavailable: {type(ex).__name__}: {str(ex)[:250]}')
            continue
        for rid in range(10):
            if fallback[i][rid]>=0:continue
            try:
                # Full two-stage search with arrival heading. If no complete
                # route exists, deliver to VIA, not to a random landmark.
                route=_full_route_option(sc,rid,opts['via'],opts['goal'],mode='fallback')
                if route is None and opts['via']:
                    route=_full_route_option(sc,rid,opts['via'],[],mode='fallback_partial')
                    if route:reasons[i].append(f'R{rid} fallback_via_only_partial_goal_unreachable')
                if route is not None and route['action'] in legal_actions(sc,rid):
                    fallback[i][rid]=route['action']
                    reasons[i].append(f'R{rid} fallback_{route["objective"]} cost={route["cost"]}')
                    continue
                # Only when no goal/via route exists: legal neighbor fallback.
                action=_legal_local_recovery(sc,rid,opts['via'] or opts['goal'])
                if action is not None and action in legal_actions(sc,rid):
                    fallback[i][rid]=action
                    reasons[i].append(f'R{rid} fallback_legal_neighbor_no_route')
                else:reasons[i].append(f'R{rid} physically_stuck')
            except (KeyboardInterrupt,SystemExit):raise
            except Exception as ex:
                reasons[i].append(f'R{rid} fallback_exception: {type(ex).__name__}: {str(ex)[:160]}')

    count=Counter()
    for lines in reasons:
        for line in lines:
            if 'fallback_via_only' in line:count['via_only']+=1
            if 'fallback_via_goal' in line:count['via_goal']+=1
            if 'fallback_goal_only' in line:count['goal_only']+=1
            if 'strict_tie_total_route' in line:count['strict_tie_total_route']+=1
            if 'fallback_legal_neighbor_no_route' in line:count['local_legal']+=1
            if 'physically_stuck' in line:count['physically_stuck']+=1
    report={'scenes':n,
            'strict_minus1':sum(a<0 for row in strict for a in row),
            'fallback_minus1':sum(a<0 for row in fallback for a in row),
            'strict_scenes_with_minus1':sum(any(a<0 for a in row) for row in strict),
            'fallback_scenes_with_minus1':sum(any(a<0 for a in row) for row in fallback),
            'recovered':sum(a<0 and b>=0 for x,y in zip(strict,fallback) for a,b in zip(x,y)),
            'modified_good_actions':sum(a>=0 and b!=a for x,y in zip(strict,fallback) for a,b in zip(x,y)),
            'recovery_types':dict(count),
            'reason_counts':Counter(line.split(':',1)[0] for lines in reasons for line in lines).most_common(35)}
    assert report['modified_good_actions']==0
    return {'strict':strict,'fallback':fallback,'reasons':reasons,
            'repairs':repairs,'report':report}


## 6c. SMART RECOVERY: multi-stage, per-robot fallback (applied after original routing definitions)


In [11]:
# ===============================================================
# V18 SMART RECOVERY — PATCH AFTER ROUTING BRIDGE, BEFORE RUN_PIPELINE
# Runs on CPU or GPU. Requires existing V18 bridge functions.
# STRlCT: frozen V18 on structurally safe scenes; explicit -1 otherwise.
# FALLBACK: per-robot mission-preserving degradation, never overwrites
# a legal strict decision. STRICT keeps -1 on missing named types; FALLBACK substitutes a seeded-random observed landmark.
# ===============================================================
from collections import Counter
import math
import hashlib
import random
import json

SMART_FALLBACK_VERSION = 'V18-SMART-RANDOM-PROXY-2026-10-10'
SMART_PROXY_RANDOM_SEED = 20261010


def _sf_requested_via(m):
    m = m or {}
    ref = m.get('via_ref') or {}
    return bool(m.get('via_required', m.get('via') not in (None, '', 'none')
                      or ref.get('kind') not in (None, '', 'none')))


def _sf_named_missing(scene, mission, role):
    name = (mission or {}).get(role)
    return name in PLACES and len(lms_of(scene, name)) == 0


def _sf_random_observed_proxy(scene, mission, role, previously_chosen=()):
    """Choose ONE *observed* substitute landmark, using a stable seeded RNG.

    Randomness is reproducible across Python kernels and reruns. Prefer a
    landmark reachable through the *observed directed graph* when available;
    we do not fabricate landmarks or graph edges. A reachable candidate is
    selected randomly, not deterministically by Manhattan distance.
    """
    pool = sorted(lms_of(scene), key=lambda lm: (lm['type'], rc(lm)))
    if not pool:
        return None, None
    avoid = set(previously_chosen)
    distinct = [lm for lm in pool if rc(lm) not in avoid]
    if distinct:
        pool = distinct

    robot = scene.get('robot') or {}
    start = None
    try:
        if 'rc' in robot:
            start = rc(robot)
    except (GroundingError, ValueError, TypeError):
        pass

    if start is not None:
        other_places = [lm for lm in pool if rc(lm) != start]
        if other_places:
            pool = other_places
        try:
            adj = allowed_edges(scene, 4)  # union-like reachability (R4 permits stairs)
            reached = {start}
            queue = [start]
            for u in queue:
                for v, _, _ in adj.get(u, []):
                    if v not in reached:
                        reached.add(v)
                        queue.append(v)
            reachable = [lm for lm in pool if rc(lm) in reached]
            if reachable:
                pool = reachable
        except (KeyboardInterrupt, SystemExit):
            raise
        except Exception:
            pass  # no graph info: random over YOLO-observed landmarks

    signature = {
        'seed': SMART_PROXY_RANDOM_SEED,
        'scene_id': str(scene.get('scene_id', '')),
        'role': role,
        'mission': mission or {},
        'landmarks': [(x['type'], list(rc(x))) for x in pool],
    }
    packed = json.dumps(signature, sort_keys=True, ensure_ascii=False, default=str)
    seed = int.from_bytes(hashlib.sha256(packed.encode('utf-8')).digest()[:8], 'big')
    chosen = random.Random(seed).choice(pool)
    return chosen, seed


def _sf_recover_options(scene, mission):
    """STRICT is unchanged; FALLBACK randomly substitutes absent named types.

    If NLU requests a named place but YOLO sees none of that type, use a
    reproducibly-random *different observed type* for that endpoint. The
    choice is a guess for submission, never claimed as the real NLU target.
    If no landmarks exist, ordinary legal-step fallback is still available.
    """
    m = mission or {}
    opts, info = _recover_options(scene, m, soft=True)
    selected = []
    for role in ('via', 'goal'):
        if _sf_named_missing(scene, m, role):
            original_type = m.get(role)
            chosen, seed = _sf_random_observed_proxy(scene, m, role, selected)
            if chosen is None:
                opts[role] = []
                info[role].update({
                    'reason': 'named_type_missing_no_landmark',
                    'coordinates': [], 'candidates': 0, 'repaired': True,
                    'random_proxy': False, 'requested_type': original_type,
                })
            else:
                proxy = {'type': chosen['type'], 'rc': list(rc(chosen))}
                opts[role] = [proxy]
                selected.append(rc(proxy))
                info[role].update({
                    'reason': 'missing_named_type_random_observed_proxy',
                    'coordinates': [proxy['rc']], 'candidates': 1,
                    'repaired': True, 'random_proxy': True,
                    'requested_type': original_type,
                    'replacement_type': proxy['type'],
                    'replacement_rc': proxy['rc'],
                    'random_seed': seed,
                })
    return opts, info


def _sf_hops_to_any(adj, start, targets):
    """Directed BFS: supports one-way/stair legality through robot's adj."""
    if not targets:
        return math.inf
    targets = set(targets)
    if start in targets:
        return 0
    seen = {start}
    todo = [(start, 0)]
    for u, n in todo:
        for v, _, _ in adj.get(u, []):
            if v in seen:
                continue
            if v in targets:
                return n + 1
            seen.add(v)
            todo.append((v, n + 1))
    return math.inf


def _sf_local_progress(scene, rid, via, goal):
    """If no complete route: prefer any legal step toward remaining mission.

    Does not hallucinate missing graph edges. Ranking:
      reachable to via (if known) > reachable to goal > reachable to any
      landmark > least Manhattan distance > local edge cost > turn preference.
    If no landmark survives, choose cheapest legal step with heading preference.
    """
    rob = scene.get('robot') or {}
    if 'rc' not in rob:
        return None, 'robot_position_missing'
    here = rc(rob)
    adj = _prepared_graph(scene, rid)
    neigh = adj.get(here, [])
    if not neigh:
        return None, 'no_legal_neighbor'
    via_rc = {rc(x) for x in via}
    goal_rc = {rc(x) for x in goal}
    lm_rc = {rc(x) for x in lms_of(scene)}
    hd = HEADING.get(str(rob.get('heading', 'UP')).upper(), 0)
    mission = scene.get('mission') or {}
    weather = scene.get('weather', 'dry')

    def k(e):
        dest, direction, ed = e
        # Goal as backup ONLY if no legal route toward via.
        dh_v = _sf_hops_to_any(adj, dest, via_rc)
        dh_g = _sf_hops_to_any(adj, dest, goal_rc)
        dh_l = _sf_hops_to_any(adj, dest, lm_rc)
        # Prioritize targets that can actually be reached from this neighbor.
        if via_rc and math.isfinite(dh_v):
            tier, hops, chosen = 0, dh_v, via_rc
        elif goal_rc and math.isfinite(dh_g):
            tier, hops, chosen = 1, dh_g, goal_rc
        elif lm_rc and math.isfinite(dh_l):
            tier, hops, chosen = 2, dh_l, lm_rc
        elif via_rc:
            tier, hops, chosen = 3, math.inf, via_rc
        elif goal_rc:
            tier, hops, chosen = 4, math.inf, goal_rc
        else:
            tier, hops, chosen = 5, math.inf, lm_rc
        mdist = min((md(dest, target) for target in chosen), default=0)
        edgecost = _edge_cost(ed, rid, weather,
                              bool(mission.get('urgent', False)),
                              bool(mission.get('fragile', False)))
        turn = _relative_rank(hd, direction)
        # Avoid stepping directly into a dead end if alternatives exist.
        dead_end = int(len(adj.get(dest, [])) <= 1)
        return (tier, hops, mdist, dead_end, edgecost, turn, direction)
    dest, act, _ = min(neigh, key=k)
    return int(act), 'local_reachable_landmark_or_legal_edge'


def _sf_route_for_robot(scene, rid, opts, reasons, info=None):
    """Respect reliable waypoint before a RANDOM substitute when possible."""
    via, goal = opts.get('via', []), opts.get('goal', [])
    info = info or {}
    via_random = bool(info.get('via', {}).get('random_proxy', False))
    goal_random = bool(info.get('goal', {}).get('random_proxy', False))
    legal = legal_actions(scene, rid)
    if not legal:
        return -1, 'no_legal_action'

    if via and goal:
        # Missing GOAL but reliable VIA: first action to VIA is usually
        # better informed than selecting the VIA->random-goal complete route.
        if goal_random and not via_random:
            plans = [('via_only', via, []),
                     ('via_goal_random_proxy', via, goal),
                     ('goal_only_random_proxy', [], goal)]
        else:
            plans = [('via_goal_random_proxy' if (via_random or goal_random)
                      else 'via_goal', via, goal),
                     ('via_only' + ('_random_proxy' if via_random else ''), via, []),
                     ('goal_only' + ('_random_proxy' if goal_random else ''), [], goal)]
    elif via:
        plans = [('via_only' + ('_random_proxy' if via_random else ''), via, [])]
    elif goal:
        plans = [('goal_only' + ('_random_proxy' if goal_random else ''), [], goal)]
    else:
        plans = []

    for name, waypoints, endpoints in plans:
        route = _full_route_option(scene, rid, waypoints, endpoints,
                                   mode='smart_' + name)
        if route is not None and route['action'] in legal:
            return int(route['action']), name

    action, why = _sf_local_progress(scene, rid, via, goal)
    if action is not None and action in legal:
        return action, 'local_progress_after_unreachable_' + ('via' if via else 'goal' if goal else 'unknown')
    return -1, why


def route_results(raw_scenes, raw_missions, policy_predict, batch_size=128, debug=False):
    """Per-robot SMART fallback and honest frozen-model strict benchmark.

    Strict -1 means frozen model could not provide a trustworthy prediction.
    Even if a different robot's route is blocked, fallback still runs for this
    robot independently; a bad scene never suppresses other scenes.
    """
    import json
    if len(raw_scenes) != len(raw_missions):
        raise ValueError('raw_scenes / raw_missions length mismatch')
    n = len(raw_scenes)
    strict = [[-1] * 10 for _ in range(n)]
    fallback = [[-1] * 10 for _ in range(n)]
    reasons = [[] for _ in range(n)]
    repairs = [{} for _ in range(n)]
    good_rows, good_ids, tie_rows = {}, [], {}
    eligibility = {}

    for i, (raw, raw_m) in enumerate(zip(raw_scenes, raw_missions)):
        m = raw_m or {}
        scene, err, details = policy_strict_scene(raw, m)
        if err is not None:
            reasons[i].append('strict_ground: ' + str(err)[:250])
            if _is_pure_spatial_tie(err):
                try:
                    opts, info = _recover_options(raw, m, soft=False)
                    if not opts['goal'] or (_sf_requested_via(m) and not opts['via']):
                        raise GroundingError('missing goal or required via')
                    sc, _ = _scene_for_options(raw, m, opts['via'], opts['goal'], soft=False)
                    tie_rows[i] = (sc, opts)
                    reasons[i].append('strict_equal_spatial: graph_route_tie')
                except (ValueError, TypeError, KeyError, GroundingError) as e:
                    reasons[i].append('strict_tie_unresolved: ' + str(e)[:180])
            continue

        good_rows[i] = scene
        try:
            diag = _primary_shapes(scene)
            goal_ok = diag['goal_coords_shape'][0] > 0
            via_ok = not diag['via_required'] or diag['via_coords_shape'][0] > 0
            eligible = {
                rid for rid in range(10)
                if goal_ok and via_ok
                and bool(diag['legal_actions'][f'R{rid}'])
                and (rid == 9 or bool(diag['reachable'][f'R{rid}']))
            }
        except (KeyboardInterrupt, SystemExit):
            raise
        except Exception as e:
            eligible = set()
            reasons[i].append(f'strict_preflight_exception: {type(e).__name__}: {e}')
        eligibility[i] = eligible
        # V18 evaluates all ten robots together. Do not submit a structurally
        # unsafe scene to a frozen model with shared empty-array operations.
        if len(eligible) == 10:
            good_ids.append(i)
        else:
            for rid in range(10):
                if rid not in eligible:
                    reasons[i].append(f'R{rid} strict_unavailable_unreachable_or_empty')
            if eligible:
                reasons[i].append('strict_partial_scene_skipped_shared_V18_policy')

    preds = _run_v18_isolated(good_rows, good_ids, policy_predict,
                              batch_size, reasons, 'strict', debug)
    for i, row in preds.items():
        sc = good_rows[i]
        for rid, action in enumerate(row):
            if action in legal_actions(sc, rid) and (rid == 9 or route_reachable(sc, rid)):
                strict[i][rid] = int(action)
            else:
                reasons[i].append(f'R{rid} strict_invalid_action_or_route: {action}')

    # Exact spatial tie: no missing landmark; compare both legs per robot.
    for i, (sc, opts) in tie_rows.items():
        for rid in range(10):
            try:
                route = _full_route_option(sc, rid, opts['via'], opts['goal'],
                                           mode='strict_equal_spatial')
                if route and route['action'] in legal_actions(sc, rid):
                    strict[i][rid] = int(route['action'])
                    reasons[i].append(f'R{rid} strict_total_route_tie')
            except (KeyboardInterrupt, SystemExit):
                raise
            except Exception as e:
                reasons[i].append(f'R{rid} strict_tie_exception: {type(e).__name__}: {e}')

    for i, (raw, m) in enumerate(zip(raw_scenes, raw_missions)):
        fallback[i] = strict[i][:]
        if all(a >= 0 for a in strict[i]):
            continue
        try:
            opts, info = _sf_recover_options(raw, m or {})
            sc, dropped = _scene_for_options(raw, m or {},
                                             opts['via'], opts['goal'], soft=True)
            repairs[i] = {'spatial': info, 'dropped_edges': dropped}
        except (KeyboardInterrupt, SystemExit):
            raise
        except Exception as e:
            reasons[i].append('fallback_scene_unavailable: '
                              + f'{type(e).__name__}: {str(e)[:220]}')
            continue
        for rid in range(10):
            if fallback[i][rid] >= 0:
                continue
            try:
                action, why = _sf_route_for_robot(sc, rid, opts, reasons[i], info=info)
                if action >= 0 and action in legal_actions(sc, rid):
                    fallback[i][rid] = int(action)
                    reasons[i].append(f'R{rid} fallback_{why}')
                else:
                    reasons[i].append(f'R{rid} still_minus1: {why}')
            except (KeyboardInterrupt, SystemExit):
                raise
            except Exception as e:
                reasons[i].append(f'R{rid} fallback_exception: {type(e).__name__}: {e}')
                # Even a Dijkstra exception must not prevent local salvage.
                try:
                    a, why = _sf_local_progress(sc, rid, opts['via'], opts['goal'])
                    if a is not None and a in legal_actions(sc, rid):
                        fallback[i][rid] = int(a)
                        reasons[i].append(f'R{rid} fallback_exception_local_recovery')
                except (KeyboardInterrupt, SystemExit):
                    raise
                except Exception as e2:
                    reasons[i].append(f'R{rid} fallback_local_failed: {e2}')

    by_type = Counter()
    by_robot = Counter()
    for i, row in enumerate(fallback):
        for rid, act in enumerate(row):
            if strict[i][rid] < 0 and act >= 0:
                by_robot[f'R{rid}'] += 1
        for msg in reasons[i]:
            if ' fallback_' in msg:
                by_type[msg.split(' fallback_', 1)[1].split()[0]] += 1
    report = {
        'version': SMART_FALLBACK_VERSION,
        'scenes': n,
        'strict_minus1': sum(a < 0 for row in strict for a in row),
        'fallback_minus1': sum(a < 0 for row in fallback for a in row),
        'strict_scenes_with_minus1': sum(any(a < 0 for a in row) for row in strict),
        'fallback_scenes_with_minus1': sum(any(a < 0 for a in row) for row in fallback),
        'recovered': sum(a < 0 <= b for x, y in zip(strict, fallback)
                         for a, b in zip(x, y)),
        'modified_good_actions': sum(a >= 0 and b != a for x, y in zip(strict, fallback)
                                     for a, b in zip(x, y)),
        'recovery_by_robot': dict(by_robot),
        'recovery_types': dict(by_type),
        'reason_counts': dict(Counter(
            msg.split(':', 1)[0] for group in reasons for msg in group)),
    }
    assert report['modified_good_actions'] == 0, 'modified frozen strict action'
    return {'strict': strict, 'fallback': fallback, 'reasons': reasons,
            'repairs': repairs, 'report': report}

print('[V18 RANDOM PROXY] Installed. Strict unchanged; missing named types use reproducible random observed landmarks.')


[V18 RANDOM PROXY] Installed. Strict unchanged; missing named types use reproducible random observed landmarks.


## 7. Observations streaming, deduplication and resize cache

In [12]:
# =====================================================================
# RESIZE 1024  (y hệt build_split lúc train: cv2.imread -> resize INTER_AREA -> lưu .jpg)
# =====================================================================
def resize_to_1024(src_path, dst_path):
    img = cv2.imread(src_path)
    if img is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {src_path}")
    cv2.imwrite(dst_path, cv2.resize(img, (IMGSZ, IMGSZ), interpolation=cv2.INTER_AREA))
    return img.shape[:2]


# =====================================================================
# ĐỌC observations.json (tự dò khoá id / ảnh / câu nhiệm vụ)
# =====================================================================
_ID_KEYS   = ('scene_id', 'id', 'obs_id', 'observation_id', 'sample_id', 'uid', 'name')
_IMG_KEYS  = ('image', 'image_path', 'img', 'img_path', 'file', 'filename', 'file_name', 'path')
_TEXT_KEYS = ('mission_text', 'text', 'instruction', 'command', 'mission', 'nlu_text', 'query', 'prompt', 'description')
_IMG_EXT   = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')

def _as_records(data):
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for k in ('observations', 'data', 'items', 'samples', 'scenes', 'records'):
            if isinstance(data.get(k), list):
                return data[k]
        return [({**v, '_key': k} if isinstance(v, dict) else {'_key': k, 'text': v}) for k, v in data.items()]
    raise ValueError("observations.json có cấu trúc lạ (không phải list/dict)")

def _first_str(d, keys):
    for k in keys:
        v = d.get(k)
        if isinstance(v, str) and v.strip():
            return v, k
        if isinstance(v, dict):                                   # vd "mission": {"text": "..."}
            for kk in ('text', 'mission_text', 'instruction'):
                if isinstance(v.get(kk), str) and v[kk].strip():
                    return v[kk], f"{k}.{kk}"
    return None, None

def _get_key(d, key):                                              # hỗ trợ khoá lồng "mission.text"
    for part in key.split('.'):
        d = d[part]
    return d

def extract_fields(o, i):
    # --- ảnh
    if OBS_IMAGE_KEY:
        img = str(_get_key(o, OBS_IMAGE_KEY))
    else:
        img, _ = _first_str(o, _IMG_KEYS)
        if img is None or not img.lower().endswith(_IMG_EXT):
            img = next((v for v in o.values() if isinstance(v, str) and v.lower().endswith(_IMG_EXT)), img)
    # --- câu nhiệm vụ
    if OBS_TEXT_KEY:
        text = str(_get_key(o, OBS_TEXT_KEY))
    else:
        text, _ = _first_str(o, _TEXT_KEYS)
        if text is None:                                           # fallback: chuỗi dài nhất có khoảng trắng, không phải đường dẫn ảnh
            cands = [v for v in o.values() if isinstance(v, str) and ' ' in v and not v.lower().endswith(_IMG_EXT)]
            text = max(cands, key=len) if cands else None
    # --- id
    if OBS_ID_KEY:
        sid = str(_get_key(o, OBS_ID_KEY))
    else:
        sid, _ = next(((str(o[k]), k) for k in _ID_KEYS if isinstance(o.get(k), (str, int)) and not isinstance(o.get(k), bool)),
                      (None, None))
        if sid is None:
            sid = str(o.get('_key') or (os.path.splitext(os.path.basename(img))[0] if img else i))
    if img is None:
        raise KeyError(f"Không tìm thấy trường ảnh trong observation #{i}: khoá có = {list(o.keys())}. Hãy đặt OBS_IMAGE_KEY.")
    if text is None:
        raise KeyError(f"Không tìm thấy câu nhiệm vụ trong observation #{i}: khoá có = {list(o.keys())}. Hãy đặt OBS_TEXT_KEY.")
    return sid, img, text

def resolve_image(split_dir, ref):
    stem = os.path.splitext(os.path.basename(ref))[0]
    for p in (os.path.join(split_dir, ref), os.path.join(split_dir, 'images', os.path.basename(ref))):
        if os.path.isfile(p):
            return p
    hits = sorted(glob.glob(os.path.join(split_dir, 'images', stem + '.*')))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Không thấy ảnh '{ref}' trong {split_dir}")

# observations.json có 1 dòng / (ảnh, robot): vd "validation-00000-R0" ... "-R9" -> cùng 1 ảnh + cùng 1 câu nhiệm vụ.
# => GỘP theo ảnh: YOLO / NLU / dựng đồ thị chỉ chạy 1 LẦN / ảnh; 10 robot chỉ khác nhau ở id (vòng for r in range(10) trong first_steps).
_ROBOT_SUFFIX = re.compile(r'[-_ ]?R(?:obot)?[-_ ]?(\d+)$', re.I)

# ==============================================================
# One-pass observations parsing. No repeated json.load on large files.
# Raw text lives only as temporary _text until NLU finishes.
# ==============================================================
import hashlib, time, ijson

def iter_observations(split_dir):
    path=Path(split_dir)/'observations.json'
    if not path.is_file():raise FileNotFoundError(path)
    with path.open('rb') as f:
        header=f.read(262144)
    h=header.lstrip()
    if h.startswith(b'['):
        with path.open('rb') as f:
            for x in ijson.items(f,'item'):yield x
        return
    if h.startswith(b'{'):
        match=re.search(rb'"(observations|data|items|samples|scenes|records)"\s*:\s*\[', header)
        if match:
            key=match.group(1).decode()
            with path.open('rb') as f:
                for x in ijson.items(f,key+'.item'):yield x
            return
        # Legacy dict keyed by scene ID: stream key/value pairs without json.load.
        with path.open('rb') as f:
            for key,value in ijson.kvitems(f,''):
                if isinstance(value,dict):yield {**value,'_key':key}
                elif isinstance(value,str):yield {'_key':key,'text':value}
                else:raise TypeError(f'Unsupported observation value at {key!r}')
        return
    raise ValueError('Unsupported JSON root in observations.json')

def load_records(split_dir,limit=None):
    records=collections.OrderedDict()
    counts=0
    for i,o in enumerate(tqdm(iter_observations(split_dir),desc='Stream observations',unit='row')):
        if not isinstance(o,dict):raise TypeError('observation must be object')
        sid,img,mission_text=extract_fields(o,i)
        # Original uses image basename as one-scene key. Include relative subdir
        # to avoid collisions across separate image folders.
        name=str(img).replace('\\','/')
        key=name.removeprefix('./').replace('images/','',1) if name.startswith('images/') else name
        k=str(key)
        g=records.get(k)
        if g is None:
            if limit is not None and len(records)>=limit:break
            g={'scene_id':_ROBOT_SUFFIX.sub('',sid),'image':img,'obs_ids':[sid],
               'idx':len(records),'_text':mission_text,'warnings':[], 'steps':[-1]*10}
            if o.get('style') in ('classic','night','sketch','print'):g['style']=o['style']
            records[k]=g
        else:
            if g['_text']!=mission_text:
                raise ValueError(f'Conflicting mission text for same image {img!r}; cannot safely merge')
            g['obs_ids'].append(sid)
        counts+=1
    recs=list(records.values())
    for r in tqdm(recs,desc='Locate source images',unit='image'):

        try: r['img_path']=resolve_image(split_dir,r['image'])
        except (FileNotFoundError,OSError) as ex:
            r['img_path']=None;r['warnings'].append('Source image missing: '+str(ex))
    return recs

def ensure_resized(r, resized_dir=RESIZED_DIR):
    source=Path(r['img_path']); st=source.stat()
    cachekey=hashlib.blake2s(f'{source.resolve()}|{st.st_size}|{st.st_mtime_ns}|{IMGSZ}'.encode(),digest_size=12).hexdigest()
    dest=Path(resized_dir)/f'{cachekey}.jpg'
    dest.parent.mkdir(parents=True,exist_ok=True)
    if not dest.is_file() or dest.stat().st_size<100:
        resize_to_1024(str(source),str(dest))
    r['resized']=str(dest)
    return str(dest)


## 8. Cache, shared YOLO/NLU inference, dual-mode robot evaluation

In [13]:
# ==============================================================
# End-to-end inference with resumable NLU/YOLO caches and visible progress
# No second observations.json read; no task text written to disk/output.
# ==============================================================
import time
from collections import Counter

def cache_signature(file_path):
    p=Path(file_path)
    stat=p.stat() if p.exists() else None
    return f'{p}|{stat.st_size if stat else "no"}|{stat.st_mtime_ns if stat else "no"}'

NLU_SIG=cache_signature(NLU_CKPT)
YOLO_SIG=cache_signature(YOLO_PATH)

def _cache_file(kind,key):
    d=CACHE_DIR/kind;d.mkdir(parents=True,exist_ok=True)
    return d/(hashlib.sha256(key.encode('utf8')).hexdigest()+'.json')

def _read_cache(p):
    try:return json.loads(p.read_text(encoding='utf8')) if p.is_file() else None
    except (json.JSONDecodeError, OSError):return None

def _atomic_json(p,obj):
    tmp=p.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False),encoding='utf8')
    tmp.replace(p)

def nlu_stage(recs):
    indices=[]; batch_texts=[]; keyfiles=[]; hit=0
    def flush():
        nonlocal indices,batch_texts,keyfiles
        if not indices:return
        pred=parse_missions(batch_texts,tta=USE_TTA)
        if len(pred)!=len(indices):raise RuntimeError('NLU batch length mismatch')
        for j,m,path in zip(indices,pred,keyfiles):
            recs[j]['nlu']=m
            _atomic_json(path,m)  # only decoded head values; never mission text
        indices=[]; batch_texts=[]; keyfiles=[]
    for i in tqdm(range(len(recs)),desc='Phase10 NLU',unit='scene'):
        r=recs[i]
        if 'nlu' in r:continue
        if '_text' not in r:raise RuntimeError('Original text unavailable; call load_records() again')
        text=r['_text']
        path=_cache_file('nlu',NLU_SIG+'|'+str(USE_TTA)+'|'+text)
        cached=_read_cache(path)
        if cached is not None: r['nlu']=cached;hit+=1
        else:
            indices.append(i);batch_texts.append(text);keyfiles.append(path)
            if len(indices)>=NLU_BATCH:flush()
        del r['_text']
    flush()




In [14]:
# ==============================================
# V18 DUAL STRICT / FALLBACK E2E RUNNER
# ==============================================
import time,hashlib,json,traceback
from collections import Counter,deque
from pathlib import Path
import numpy as np

@torch.inference_mode()
def predict_yolo_batch(recs, indices):
    out={};todo=[];paths=[];cache_files=[];hit=0
    for i in indices:
        rec=recs[i]
        if not rec.get('img_path'):
            rec['warnings'].append('YOLO missing source image');continue
        try:
            pth=ensure_resized(rec)
        except Exception as ex:
            rec['warnings'].append('YOLO resize: '+type(ex).__name__+': '+str(ex)[:180]);continue
        sig=YOLO_SIG+'|'+cache_signature(pth)+'|'+str(YOLO_MIN_CONF)+'|'+str(IMGSZ)
        fn=_cache_file('yolo',sig)
        previous=_read_cache(fn)
        if previous is not None:out[i]=previous;hit+=1
        else:todo.append(i);paths.append(pth);cache_files.append(fn)
    if todo:
        gen=yolo.predict(source=paths,conf=YOLO_MIN_CONF,imgsz=IMGSZ,
                         batch=min(YOLO_BATCH,len(todo)),device=YOLO_DEVICE,
                         stream=True,verbose=False)
        count=0
        for idx,fn,res in zip(todo,cache_files,gen):
            dets=[]
            if res.boxes is not None:
                coords=res.boxes.xyxy.cpu().numpy();cls=res.boxes.cls.cpu().numpy();conf=res.boxes.conf.cpu().numpy()
                for bb,cl,cc in zip(coords,cls,conf):
                    cl=int(cl)
                    if cl<0 or cl>=len(CLASSES):continue
                    bb=[float(t) for t in bb]
                    dets.append({'class':CLASSES[cl],'box':bb,'conf':float(cc),
                                 'center':get_center(bb),'w':bb[2]-bb[0],'h':bb[3]-bb[1]})
            _atomic_json(fn,dets);out[idx]=dets;count+=1
        if count != len(todo):
            raise RuntimeError('YOLO returned '+str(count)+' images for '+str(len(todo)))
    return out,hit

def _scene_from_detections(rec,dets):
    # Multiple robot boxes on one map: keep strongest; keep ALL distinct same-type landmarks.
    poses=[d for d in dets if d['class'].startswith('robot_')]
    if poses:
        pose=max(poses,key=lambda d:d['conf'])
        dets=[d for d in dets if not d['class'].startswith('robot_')]+[pose]
    scene=dets_to_scene(dets,conf=YOLO_CONF,edge_conf=None,txt_side='right',split_long=True)
    scene['scene_id']=rec['scene_id']
    scene['landmarks']=lms_of(scene)
    if rec.get('style') in ('classic','night','sketch','print'):scene['style']=rec['style']
    return scene

def _empty_scene(rec):
    return {'scene_id':rec['scene_id'],'robot':{},'nodes':[],'edges':[],
            'landmarks':[],'weather':'dry','warnings':['YOLO failed or image missing']}

def run_pipeline(split_dir,recs=None,limit=None,save=True):
    """Runs exactly ONE YOLO and ONE NLU pass, then BOTH V18 policy modes.
    Returns records with strict_steps, fallback_steps; 'steps' aliases fallback for old exporter.
    """
    global LAST_DUAL_RESULT
    t_all=time.monotonic()
    if recs is None:recs=load_records(split_dir,limit=limit)
    elif limit is not None:recs=recs[:limit]
    if not recs:raise ValueError('No observations found')
    t=time.monotonic();nlu_stage(recs)
    raw_scenes=[_empty_scene(r) for r in recs]
    hits=0;t=time.monotonic()
    recent=deque(maxlen=8)
    for bi,st in enumerate(tqdm(range(0,len(recs),YOLO_BATCH),desc='YOLO → graph',unit='batch'),1):
        btime=time.monotonic()
        inds=list(range(st,min(st+YOLO_BATCH,len(recs))))
        try:
            result,h=predict_yolo_batch(recs,inds);hits+=h
        except KeyboardInterrupt:
            pass;raise
        except Exception as ex:
            # One bad batch: retry each image just once, not robot recursion.
            result={}
            for idx in inds:
                try:
                    one,h=predict_yolo_batch(recs,[idx]);result.update(one);hits+=h
                except KeyboardInterrupt:raise
                except Exception as exc:recs[idx]['warnings'].append('YOLO: '+repr(exc)[:200])
        for idx,dets in result.items():
            try:
                raw_scenes[idx]=_scene_from_detections(recs[idx],dets)
            except KeyboardInterrupt:raise
            except Exception as e:
                recs[idx]['warnings'].append('Graph: '+repr(e)[:220])
        recent.append(time.monotonic()-btime)
        if bi<=2 or bi%10==0 or st+len(inds)==len(recs):
            speed=len(inds)/max(.001,np.mean(recent));remain=max(0,len(recs)-st-len(inds))
    t=time.monotonic()
    # Raw NLU ('none' + spatial kind) is passed to grounding. Never pass oracle resolved mission.
    raw_missions=[r.get('nlu') or {} for r in recs]
    dual=route_results(raw_scenes,raw_missions,predict_v18_batch,batch_size=ROBOT_V18_BATCH)
    for i,r in enumerate(recs):
        r['strict_steps']=dual['strict'][i]
        r['fallback_steps']=dual['fallback'][i]
        r['steps']=dual['fallback'][i]
        r['warnings']+=dual['reasons'][i]
    LAST_DUAL_RESULT=dual
    if save:save_dual_outputs(recs,split_dir,dual)
    return recs


## 9. Submission formatter; strict -1 files always retained

In [15]:
def _lookup(records):
    lk = {}
    for r in records:
        b = os.path.basename(str(r['image']))
        for key in (str(r['scene_id']), str(r['image']), b, os.path.splitext(b)[0]):
            lk.setdefault(key, r['idx'])
    return lk

def build_submission(sample, records):
    acts = [r['steps'] for r in records]; N = len(records); lk = _lookup(records)
    flat = [a for row in acts for a in row]

    def idx_of(row, i):
        for v in row.values():
            if isinstance(v, (str, int)) and not isinstance(v, bool):
                for cand in (str(v), _ROBOT_SUFFIX.sub('', str(v))):      # "validation-00000-R3" -> "validation-00000"
                    if cand in lk:
                        return lk[cand]
        return i

    def robot_of(row, j):
        rk = next((k for k in row if 'robot' in str(k).lower() and isinstance(row[k], int) and not isinstance(row[k], bool)), None)
        if rk is not None: return row[rk]
        for v in row.values():                                              # robot suy từ hậu tố của id: "...-R3"
            m = isinstance(v, str) and _ROBOT_SUFFIX.search(v)
            if m: return int(m.group(1))
        return j % 10

    def fill_row(row, a):
        row = dict(row)
        for k, v in row.items():
            if isinstance(v, list) and len(v) == 10:
                row[k] = list(a); return row
            if isinstance(v, dict) and len(v) == 10:
                row[k] = {kk: a[int(re.findall(r'\d+', str(kk))[-1]) if re.findall(r'\d+', str(kk)) else j]
                          for j, kk in enumerate(v.keys())}
                return row
        rk = [k for k in row if re.fullmatch(r'(robot|bot|r)[_\- ]?\d+', str(k), re.I)]
        if len(rk) == 10:
            for k in rk: row[k] = a[int(re.findall(r'\d+', k)[-1])]
            return row
        return None

    # (1) list phẳng các số nguyên
    if isinstance(sample, list) and sample and all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in sample):
        return flat, f"list phẳng {N}x10 (giống labels.json); sample có {len(sample)} phần tử"
    # (2) list các dict
    if isinstance(sample, list) and sample and isinstance(sample[0], dict):
        if len(sample) == 10 * N and len(sample) != N:                       # mỗi dòng = 1 (scene, robot)
            out = []
            for j, row in enumerate(sample):
                row = dict(row); i = idx_of(row, j // 10)
                ak = next((k for k in row if str(k).lower() in _ACT_KEYS), None)
                rid = robot_of(row, j)
                if ak is None: out = None; break
                row[ak] = acts[i][rid]; out.append(row)
            if out is not None:
                return out, "list dict, mỗi dòng = 1 (scene, robot)"
        else:
            out = []
            for i, row in enumerate(sample):
                new = fill_row(row, acts[idx_of(row, i)])
                if new is None: out = None; break
                out.append(new)
            if out is not None:
                return out, "list dict, mỗi dòng = 1 scene (10 bước đi trong 1 trường)"
    # (3) dict id -> 10 số / dict robot -> số
    if isinstance(sample, dict):
        lists = [k for k, v in sample.items() if isinstance(v, list)]
        if len(lists) == 1 and sample[lists[0]] and isinstance(sample[lists[0]][0], (dict, int)):   # {"predictions": [...]}
            inner, desc = build_submission(sample[lists[0]], records)
            return {**sample, lists[0]: inner}, f"dict bọc ngoài khoá '{lists[0]}' -> {desc}"
        if sample and all(str(k) in lk for k in sample):
            out = {}
            for k, v in sample.items():
                a = acts[lk[str(k)]]
                out[k] = ({kk: a[int(re.findall(r'\d+', str(kk))[-1]) if re.findall(r'\d+', str(kk)) else j]
                           for j, kk in enumerate(v.keys())} if isinstance(v, dict) else list(a))
            return out, "dict id -> 10 bước đi"
    # (4) không nhận ra -> dict id -> 10 số
    return {str(r['scene_id']): r['steps'] for r in records}, "KHÔNG nhận ra cấu trúc sample_submission -> dùng dict {scene_id: [10 số]} (đã có thêm submission_flat.json)"


def save_dual_outputs(records,split_dir,dual):
    """Create prediction outputs in both modes; strict may retain -1.
    Never fill invalid actions with invented values solely to satisfy a submission.
    """
    output_dir=Path('/content/pkaa_v18_results')
    output_dir.mkdir(parents=True,exist_ok=True)
    sample_path=Path(split_dir)/'sample_submission.json'
    sample=json.loads(sample_path.read_text(encoding='utf8')) if sample_path.exists() else None
    for mode in ('strict','fallback'):
        name='strict_steps' if mode=='strict' else 'fallback_steps'
        rows=[{**r,'steps':list(r[name])} for r in records]
        flat=[int(a) for r in rows for a in r['steps']]
        (output_dir/f'{mode}_flat.json').write_text(json.dumps(flat,ensure_ascii=False),encoding='utf8')
        fmt,desc=build_submission(sample,rows) if sample is not None else (flat,'flat scene-major R0..R9')
        (output_dir/f'{mode}_submission_format.json').write_text(json.dumps(fmt,ensure_ascii=False),encoding='utf8')
    if all(x>=0 for r in records for x in r['fallback_steps']):
        pass
    else:
        pass
    return output_dir


## 10. Run manually — test 5 scenes first

In [16]:
# Run full inference and save predictions.
records = run_pipeline(TEST_DIR, save=True)


In [17]:
# ============================================================
# FALLBACK V2 — ENSEMBLE GRID REPAIR + EVIDENCE RESCUE + VOTING
# Không hiểu lắm, đề kêu grid 5-9 hàng/cột thế mà có cái đếch nằm trong
#
# Nguyên tắc:
#   * Chỉ đụng tới robot có strict = -1. Không thay đổi bất kỳ hành động strict hợp lệ nào.
#   * Không có quy tắc riêng cho scene_id nào. Mọi sửa chữa kích hoạt theo ĐIỀU KIỆN LỖI.
#   * Ưu tiên để V18 thật quyết định. Heuristic chỉ dùng khi V18 không chạy được.
#
# Các tầng:
#   1. Sinh nhiều biến thể đồ thị từ CÙNG một bộ detection YOLO (cache, conf>=0.1):
#        - đổi tolerance phân cụm lưới, bỏ hàng/cột 'ma'
#        - bỏ landmark conf thấp
#        - gộp hàng/cột liền kề khi lưới > 9 (không có cạnh nối giữa 2 hàng đó)
#        - CỨU BẰNG BẰNG CHỨNG: thiếu robot -> lấy box robot conf thấp nhất định;
#          NLU cần loại landmark mà YOLO không thấy ở 0.5 -> tìm box loại đó ở conf >= 0.2
#   2. Chạy V18 thật trên mọi biến thể hợp lệ (1 lần batch).
#   3. BỎ PHIẾU theo từng robot. Biến thể trùng nhau từ nhiều đường sửa khác nhau
#      được cộng thêm phiếu (nhiều cách sửa độc lập cùng ra 1 đồ thị = bằng chứng mạnh).
#   4. Không biến thể nào chạy được -> Dijkstra đầy đủ trên đồ thị gốc (không giới hạn 9x9).
# ============================================================
import json, math
from copy import deepcopy
from collections import Counter
from pathlib import Path

if '_ORIG_dynamic_cluster' not in globals():
    _ORIG_dynamic_cluster = dynamic_cluster

EVIDENCE_MIN_CONF = 0.20      # ngưỡng tìm lại landmark/robot bị YOLO bỏ sót
MAX_VARIANTS_PER_SCENE = 48


# ---------------- tiện ích lưới ----------------
def _fb_cluster(tol_ratio):
    def _dc(coords, unit_size):
        if not coords:
            return []
        coords = sorted(coords)
        cl = [[coords[0]]]
        for c in coords[1:]:
            if c - np.mean(cl[-1]) <= unit_size * tol_ratio:
                cl[-1].append(c)
            else:
                cl.append([c])
        return [float(np.mean(x)) for x in cl]
    return _dc


def _fb_points(sc):
    pts = [tuple(lm['rc']) for lm in sc.get('landmarks', [])]
    if (sc.get('robot') or {}).get('rc') is not None:
        pts.append(tuple(sc['robot']['rc']))
    for e in sc.get('edges', []):
        pts += [tuple(e['a']), tuple(e['b'])]
    return pts


def _fb_dims(sc):
    pts = _fb_points(sc)
    return (max(p[0] for p in pts) + 1, max(p[1] for p in pts) + 1) if pts else (0, 0)


def _fb_remap(sc, fr, fc):
    s = deepcopy(sc)
    f = lambda p: [fr(int(p[0])), fc(int(p[1]))]
    for lm in s.get('landmarks', []):
        lm['rc'] = f(lm['rc'])
    if (s.get('robot') or {}).get('rc') is not None:
        s['robot']['rc'] = f(s['robot']['rc'])
    merged = {}
    for e in s.get('edges', []):
        a, b = f(e['a']), f(e['b'])
        if a == b:
            continue
        e['a'], e['b'] = a, b
        if e.get('oneway_to') is not None:
            e['oneway_to'] = f(e['oneway_to'])
        key = tuple(sorted((tuple(a), tuple(b))))
        if key in merged:   # 2 cạnh trùng sau khi gộp: giữ trạng thái chặt hơn
            old = merged[key]
            rank = {'closed': 3, 'crowded': 2, 'covered': 1, 'normal': 0}
            if rank.get(e.get('status'), 0) > rank.get(old.get('status'), 0):
                old['status'] = e.get('status')
            old['stairs'] = bool(old.get('stairs')) or bool(e.get('stairs'))
            old['oneway_to'] = old.get('oneway_to') or e.get('oneway_to')
        else:
            merged[key] = e
    s['edges'] = list(merged.values())
    s['landmarks'] = lms_of(s)
    R, C = _fb_dims(s)
    s['nodes'] = [{'rc': [r, c]} for r in range(R) for c in range(C)]
    s.pop('grid', None)
    return s


def _fb_compact(sc):
    pts = _fb_points(sc)
    if not pts:
        return sc
    rmap = {r: i for i, r in enumerate(sorted({p[0] for p in pts}))}
    cmap = {c: i for i, c in enumerate(sorted({p[1] for p in pts}))}
    return _fb_remap(sc, lambda r: rmap[r], lambda c: cmap[c])


def _fb_merge_variants(sc, axis):
    """Gộp hàng/cột k với k+1 khi KHÔNG có cạnh nào nối giữa chúng (dấu hiệu 1 hàng bị tách đôi)."""
    n = _fb_dims(sc)[axis]
    out = []
    for k in range(n - 1):
        crossing = any(e['a'][axis] != e['b'][axis] and {e['a'][axis], e['b'][axis]} == {k, k + 1}
                       for e in sc.get('edges', []))
        if crossing:
            continue
        g = lambda i, k=k: i - 1 if i > k else i
        out.append((k, _fb_remap(sc, g, lambda c: c) if axis == 0 else _fb_remap(sc, lambda r: r, g)))
    return out


# ---------------- dựng scene từ detection ----------------
def _fb_build(rec, dets, tol):
    globals()['dynamic_cluster'] = _fb_cluster(tol)
    try:
        return _scene_from_detections(rec, dets)
    finally:
        globals()['dynamic_cluster'] = _ORIG_dynamic_cluster


def _fb_needed_types(m):
    m = m or {}
    need = set()
    for role in ('goal', 'via'):
        if m.get(role) in PLACES:
            need.add(m[role])
        anc = (m.get(role + '_ref') or {}).get('anchor')
        if anc in PLACES:
            need.add(anc)
    return need


def _fb_evidence_dets(dets, m):
    """Thêm lại box bị cắt bởi ngưỡng 0.5 NHƯNG chỉ khi có lý do:
       - không có robot nào >= YOLO_CONF -> lấy robot conf cao nhất
       - NLU cần loại X mà không có box X >= YOLO_CONF -> lấy box X conf cao nhất >= EVIDENCE_MIN_CONF
    Trả về (dets_mới, danh_sách_lý_do) hoặc (None, []) nếu không cần cứu."""
    add, why = [], []
    robots = [d for d in dets if d['class'].startswith('robot_')]
    if robots and not any(d['conf'] >= YOLO_CONF for d in robots):
        best = max(robots, key=lambda d: d['conf'])
        if best['conf'] >= EVIDENCE_MIN_CONF:
            add.append(best); why.append(f"robot@{best['conf']:.2f}")
    for t in sorted(_fb_needed_types(m)):
        same = [d for d in dets if d['class'] == t]
        if same and not any(d['conf'] >= YOLO_CONF for d in same):
            best = max(same, key=lambda d: d['conf'])
            if best['conf'] >= EVIDENCE_MIN_CONF:
                add.append(best); why.append(f"{t}@{best['conf']:.2f}")
    if not add:
        return None, []
    ids = {id(d) for d in add}
    out = [d for d in dets if id(d) not in ids]
    for d in add:
        d2 = dict(d); d2['conf'] = max(d['conf'], YOLO_CONF + 1e-3)
        out.append(d2)
    return out, why


def _fb_variants(rec, dets, m):
    det_sets = [('base', dets)]
    ev, why = _fb_evidence_dets(dets, m)
    if ev is not None:
        det_sets.append(('evidence[' + ','.join(why) + ']', ev))
    for dname, D in det_sets:
        for tol in (0.4, 0.45, 0.5, 0.35, 0.55, 0.3, 0.6):
            try:
                base = _fb_build(rec, D, tol)
            except Exception:
                continue
            yield f'{dname}|tol{tol}', base
            comp = _fb_compact(base)
            yield f'{dname}|tol{tol}|compact', comp
            for axis, name in ((0, 'row'), (1, 'col')):
                if _fb_dims(comp)[axis] > 9:
                    for k, mv in _fb_merge_variants(comp, axis):
                        yield f'{dname}|tol{tol}|compact|merge_{name}{k}', mv
        for thr in (0.6, 0.7, 0.8):
            D2 = [d for d in D if not (d['class'] in NODE_CLS and not d['class'].startswith('robot_')
                                       and d['conf'] < thr and d['class'] not in _fb_needed_types(m))]
            for tol in (0.4, 0.5):
                try:
                    yield f'{dname}|lm>={thr}|tol{tol}|compact', _fb_compact(_fb_build(rec, D2, tol))
                except Exception:
                    continue


def _fb_key(sc):
    return json.dumps([(sc.get('robot') or {}).get('rc'), (sc.get('robot') or {}).get('heading'),
                       sorted((lm['type'], tuple(lm['rc'])) for lm in sc.get('landmarks', [])),
                       sorted((tuple(e['a']), tuple(e['b']), e.get('status'), bool(e.get('stairs')),
                               tuple(e['oneway_to']) if e.get('oneway_to') else None)
                              for e in sc.get('edges', []))], default=str)


# ---------------- tầng cuối: đồ thị gốc ----------------
def _fb_plan_raw(raw, m, rid):
    m = m or {}
    if not (raw.get('robot') or {}).get('rc'):
        return -1, 'no_robot'
    opts, _ = _recover_options(raw, m, soft=True)
    sc = deepcopy(raw)
    sc['mission'] = {'urgent': bool(m.get('urgent', False)), 'fragile': bool(m.get('fragile', False))}
    route = _full_route_option(sc, rid, opts['via'], opts['goal'], mode='raw')
    if route is None and opts['via']:
        route = _full_route_option(sc, rid, opts['via'], [], mode='raw_via')
    if route is not None and route['action'] in legal_actions(sc, rid):
        return int(route['action']), 'raw_dijkstra'
    a, why = _sf_local_progress(sc, rid, opts['via'], opts['goal'])
    return (int(a), 'raw_local') if a is not None and a in legal_actions(sc, rid) else (-1, why)


# ---------------- chương trình chính ----------------
def fallback_v2(records, verbose=True):
    bad = [i for i, r in enumerate(records) if any(a < 0 for a in r['strict_steps'])]
    uniq_scenes, uniq_missions, uniq_owner = [], [], []
    weight = {}          # (scene_idx, uniq_idx) -> số đường sửa ra cùng đồ thị
    first_order = {}
    originals = {}
    for i in bad:
        r = records[i]; m = r.get('nlu') or {}
        dets = predict_yolo_batch(records, [i])[0].get(i, [])
        try:
            originals[i] = _scene_from_detections(r, dets)
        except Exception:
            originals[i] = _empty_scene(r)
        seen = {}
        n_valid = 0
        for tag, sc in _fb_variants(r, dets, m):
            key = _fb_key(sc)
            if key in seen:
                weight[(i, seen[key])] += 1
                continue
            if len(seen) >= MAX_VARIANTS_PER_SCENE:
                break
            _, err, _ = policy_strict_scene(sc, m)
            if err is not None:
                seen[key] = None
                weight[(i, None)] = 0
                continue
            u = len(uniq_scenes)
            uniq_scenes.append(sc); uniq_missions.append(m); uniq_owner.append(i)
            seen[key] = u; weight[(i, u)] = 1; first_order[u] = (n_valid, tag)
            n_valid += 1

    per_scene = {}
    if uniq_scenes:
        out = route_results(uniq_scenes, uniq_missions, predict_v18_batch, batch_size=ROBOT_V18_BATCH)
        for u, i in enumerate(uniq_owner):
            per_scene.setdefault(i, []).append((u, out['strict'][u], out['fallback'][u]))

    changed, summary = 0, []
    for i in bad:
        r = records[i]; m = r.get('nlu') or {}
        cands = per_scene.get(i, [])
        new = list(r['strict_steps'])
        modes = []
        for rid in range(10):
            if new[rid] >= 0:
                continue
            # Ưu tiên phiếu từ V18 strict; nếu không có thì phiếu từ fallback trên đồ thị đã sửa.
            for src, idx in (('V18', 1), ('fb', 2)):
                votes = Counter(); first = {}
                for c in cands:
                    a = c[idx][rid]
                    if a in (0, 1, 2, 3):
                        w = weight[(i, c[0])]
                        votes[a] += w
                        first.setdefault(a, first_order[c[0]][0])
                if votes:
                    best = max(votes, key=lambda a: (votes[a], -first[a]))
                    new[rid] = int(best)
                    modes.append(f'R{rid}:{src}{dict(votes)}')
                    break
            if new[rid] < 0:
                a, why = _fb_plan_raw(originals[i], m, rid)
                if a in (0, 1, 2, 3):
                    new[rid] = a; modes.append(f'R{rid}:{why}')
        # an toàn tuyệt đối: không đổi strict hợp lệ
        assert all(new[k] == r['strict_steps'][k] for k in range(10) if r['strict_steps'][k] >= 0)
        if new != r['fallback_steps']:
            changed += 1
        summary.append((r['scene_id'], list(r['fallback_steps']), new, modes))
        r['fallback_steps'] = new; r['steps'] = new
        r.setdefault('warnings', []).append('fallback_v2: ' + '; '.join(modes)[:400])

    return summary


summary_v2 = fallback_v2(records, verbose=False)

# ---------- xuất submission ----------
flat = [int(a) for r in records for a in r['fallback_steps']]
bad_n = sum(a not in (0, 1, 2, 3) for a in flat)
if bad_n:   # còn sót thì lấp bằng rescue cũ (chỉ ô -1)
    rescue_remaining(records)
    flat = [int(a) for r in records for a in r['fallback_steps']]
    bad_n = sum(a not in (0, 1, 2, 3) for a in flat)
assert len(flat) == 12000 and bad_n == 0

OUT_DIR = Path('/content/pkaa_v18_results'); OUT_DIR.mkdir(parents=True, exist_ok=True)
sample_path = Path(TEST_DIR) / 'sample_submission.json'
if sample_path.exists():
    submission, fmt = build_submission(json.loads(sample_path.read_text(encoding='utf-8')), records)
else:
    submission, fmt = flat, 'flat'
out = OUT_DIR / 'submission_fallback_v2.json'
out.write_text(json.dumps(submission, ensure_ascii=False), encoding='utf-8')

try:
    shutil.copy(out, Path(DRIVE_OUT_DIR) / out.name)
except Exception as ex:
    pass

In [18]:
n_nlu  = sum(p.numel() for p in model.parameters())
n_yolo = sum(p.numel() for p in yolo.model.parameters())
print(f'NLU {n_nlu/1e6:.1f}M + YOLO {n_yolo/1e6:.1f}M = {(n_nlu+n_yolo)/1e6:.1f}M (+ V18, xem file weights)')

NLU 105.7M + YOLO 25.9M = 131.6M (+ V18, xem file weights)


File submit nằm trong /pkaa_v18_results/submission_fallback_v2.json